In [61]:
# ============================================================
# STEP 0 — SETUP
# Run this cell FIRST in Google Colab
# ============================================================

# ── Install all dependencies ─────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

packages = [
    "sentence-transformers",   # bi-encoder + cross-encoder
    "faiss-cpu",               # vector index for 1L MCATs
    "transformers",            # HuggingFace: CLIP, ViT, NLI
    "torch",                   # PyTorch backend
    "Pillow",                  # image loading
    "pandas",                  # dataframes
    "openpyxl",                # read .xlsx files
    "fastapi",                 # REST API
    "uvicorn",                 # ASGI server
    "gradio",                  # demo UI
    "google-generativeai",     # Gemini Flash for LLM reasoning
    "langdetect",              # language detection
    "nltk",                    # text utilities
    "scikit-learn",            # metrics + calibration
    "tqdm",                    # progress bars
]

for p in packages:
    install(p)

print("✅ All packages installed")

# ── Mount Google Drive (for persisting FAISS index) ──────────
# Uncomment when running in Colab:
from google.colab import drive
drive.mount('/content/drive')
INDEX_SAVE_PATH = "/content/drive/MyDrive/buylead_rca/"

# ── Project folder structure ─────────────────────────────────
import os

DIRS = [
    "data",           # MCAT dump, sample buyLeads
    "index",          # FAISS index + metadata
    "models",         # cached model weights
    "outputs",        # RCA JSON results
    "logs",           # run logs
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)

print("✅ Folder structure created")
print("""
PROJECT STRUCTURE:
  00_setup.py           ← You are here (run first)
  01_data_prep.py       ← Load MCAT dump, build search index
  02_rule_engine.py     ← Fast rule-based checks
  03_mcat_ranker.py     ← Embedding-based MCAT remapping
  04_image_analyzer.py  ← CLIP image-category consistency
  05_nlp_pipeline.py    ← NLI contradiction, title quality
  06_llm_reasoner.py    ← Gemini: narrative RCA, auto-description
  07_scorer.py          ← Weighted score fusion
  08_rca_agent.py       ← Main pipeline (orchestrates all modules)
  09_demo_gradio.py     ← Gradio UI for hackathon demo
""")

✅ All packages installed
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Folder structure created

PROJECT STRUCTURE:
  00_setup.py           ← You are here (run first)
  01_data_prep.py       ← Load MCAT dump, build search index
  02_rule_engine.py     ← Fast rule-based checks
  03_mcat_ranker.py     ← Embedding-based MCAT remapping
  04_image_analyzer.py  ← CLIP image-category consistency
  05_nlp_pipeline.py    ← NLI contradiction, title quality
  06_llm_reasoner.py    ← Gemini: narrative RCA, auto-description
  07_scorer.py          ← Weighted score fusion
  08_rca_agent.py       ← Main pipeline (orchestrates all modules)
  09_demo_gradio.py     ← Gradio UI for hackathon demo



In [62]:
# ════════════════════════════════════════════════════
# CELL — Create folders + build index FIRST
# Run this before the "execute all" cell
# ════════════════════════════════════════════════════

import os

BASE = "/content/drive/MyDrive/buylead_rca"

# Create all required folders
folders = [
    BASE,
    f"{BASE}/index",
    f"{BASE}/outputs",
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ Folder ready: {folder}")

print("\n✅ All folders created")

✅ Folder ready: /content/drive/MyDrive/buylead_rca
✅ Folder ready: /content/drive/MyDrive/buylead_rca/index
✅ Folder ready: /content/drive/MyDrive/buylead_rca/outputs

✅ All folders created


In [63]:
# ════════════════════════════════════════════════════
# CELL — Verify files exist, then build index
# ════════════════════════════════════════════════════

MCAT_FILE  = "/content/drive/MyDrive/buylead_rca/8_May_MCAT_Dump.xlsx"
INDEX_FILE = "/content/drive/MyDrive/buylead_rca/index/mcat_faiss.index"
META_FILE  = "/content/drive/MyDrive/buylead_rca/index/mcat_metadata.pkl"

# Check MCAT file exists
if not os.path.exists(MCAT_FILE):
    print(f"❌ MCAT file not found at: {MCAT_FILE}")
    print("   Upload 8_May_MCAT_Dump.xlsx to Google Drive at that path first.")
else:
    print(f"✅ MCAT file found")

# Check if index already built
if os.path.exists(INDEX_FILE) and os.path.exists(META_FILE):
    print("⚡ Index already exists — will load from disk, no rebuild needed")
else:
    print("🔄 Index not found — will be built when 01_data_prep runs")
    print("   This takes ~10 min on T4 GPU, then cached forever")

✅ MCAT file found
⚡ Index already exists — will load from disk, no rebuild needed


In [64]:
# %%writefile /content/drive/MyDrive/buylead_rca/01_data_prep.py

import os, pickle, json, pprint
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

MCAT_FILE         = "/content/drive/MyDrive/buylead_rca/8_May_MCAT_Dump.xlsx"
BUYLEAD_JSON_FILE = "/content/drive/MyDrive/buylead_rca/buyLeads.json"
INDEX_FILE        = "/content/drive/MyDrive/buylead_rca/index/mcat_faiss.index"
META_FILE         = "/content/drive/MyDrive/buylead_rca/index/mcat_metadata.pkl"
MODEL_NAME        = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
BATCH_SIZE        = 256

# ── CHANGE 1: Removed glcat_mcat_name and attribute_source ───
# Both can be null in real data — not safe to require them
REQUIRED_FIELDS = [
    "display_id",
    "eto_ofr_title",
    "eto_ofr_mcat_id",
    "specifications",
]

def load_mcat_dump(filepath):
    df = pd.read_excel(filepath)
    df.columns = [c.strip() for c in df.columns]
    for col in ["MCAT Name", "CAT Name", "GROUP Name", "PMCAT Name"]:
        if col in df.columns:
            df[col] = df[col].fillna("")
    df["full_path"] = (
        df["GROUP Name"].str.strip() + " > " +
        df["CAT Name"].str.strip()   + " > " +
        df.get("PMCAT Name", pd.Series([""] * len(df))).str.strip() + " > " +
        df["MCAT Name"].str.strip()
    )
    df["full_path"] = df["full_path"].str.replace(r"\s*>\s*>\s*", " > ", regex=True).str.strip(" >")
    print(f"✅ Loaded {len(df):,} MCATs")
    return df

def load_mcat_index():
    index = faiss.read_index(INDEX_FILE)
    with open(META_FILE, "rb") as f:
        metadata = pickle.load(f)
    model = SentenceTransformer(metadata["model_name"])
    print(f"✅ Loaded index: {index.ntotal:,} vectors")
    return index, metadata, model

def build_mcat_index(df):
    if os.path.exists(INDEX_FILE) and os.path.exists(META_FILE):
        print("⚡ Index already exists — loading from disk")
        return load_mcat_index()
    model = SentenceTransformer(MODEL_NAME)
    texts = df["full_path"].tolist()
    print(f"🔄 Encoding {len(texts):,} MCATs...")
    embeddings = model.encode(
        texts, batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    os.makedirs(os.path.dirname(INDEX_FILE), exist_ok=True)
    faiss.write_index(index, INDEX_FILE)
    metadata = {
        "mcat_ids":    df["MCAT ID"].tolist(),
        "mcat_names":  df["MCAT Name"].tolist(),
        "full_paths":  df["full_path"].tolist(),
        "cat_names":   df["CAT Name"].tolist(),
        "group_names": df["GROUP Name"].tolist(),
        "model_name":  MODEL_NAME,
        "dim":         dim,
        "total":       len(texts),
    }
    with open(META_FILE, "wb") as f:
        pickle.dump(metadata, f)
    print(f"✅ Index built: {index.ntotal:,} vectors | dim={dim}")
    return index, metadata, model

# ── CHANGE 2: Completely rewritten load_buyLeads ──────────────
# Handles: null glcat_mcat_name, null attribute_source,
#          null available_isq_at_approval, mcat_id=0
def load_buyLeads(filepath=BUYLEAD_JSON_FILE):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: '{filepath}'")

    with open(filepath, "r", encoding="utf-8") as f:
        raw = json.load(f)

    if isinstance(raw, dict):
        raw = [raw]

    buyLeads = []
    skipped  = 0

    for i, item in enumerate(raw):

        # Fix all nullable fields BEFORE any checks
        item["glcat_mcat_name"]           = str(item.get("glcat_mcat_name")           or "").strip()
        item["eto_ofr_desc"]              = str(item.get("eto_ofr_desc")              or "").strip()
        item["specifications"]            = str(item.get("specifications")            or "").strip()
        item["attribute_source"]          = str(item.get("attribute_source")          or "").strip()
        item["image_url"]                 = item.get("image_url", None)
        item["available_isq_at_approval"] = item.get("available_isq_at_approval", 0) or 0

        # Skip only if truly critical fields are missing
        missing = [
            fld for fld in REQUIRED_FIELDS
            if fld not in item or item[fld] is None
        ]
        if missing:
            print(f"  ⚠️  Row {i} (id={item.get('display_id')}): skipping — missing: {missing}")
            skipped += 1
            continue

        # Skip if title is empty string
        if not str(item.get("eto_ofr_title", "")).strip():
            print(f"  ⚠️  Row {i} (id={item.get('display_id')}): skipping — empty title")
            skipped += 1
            continue

        # Normalize types
        try:
            item["eto_ofr_mcat_id"] = int(item["eto_ofr_mcat_id"])
        except (ValueError, TypeError):
            item["eto_ofr_mcat_id"] = 0

        item["eto_ofr_title"] = str(item["eto_ofr_title"]).strip()

        buyLeads.append(item)

    print(f"✅ Loaded {len(buyLeads)} BuyLeads" + (f"  ({skipped} skipped)" if skipped else ""))
    return buyLeads

def save_sample_data():
    if os.path.exists(BUYLEAD_JSON_FILE):
        print(f"ℹ️  Already exists — not overwriting.")
        return
    os.makedirs(os.path.dirname(BUYLEAD_JSON_FILE), exist_ok=True)
    sample = [
        {
            "display_id": 144097681163,
            "glcat_mcat_name": "Cloth Bags",
            "eto_ofr_title": "Cotton Carry Bags",
            "available_isq_at_approval": 5,
            "eto_ofr_desc": "",
            "eto_ofr_mcat_id": 25024,
            "specifications": "Bag Type:Printed | Material:Cotton | Quantity:100 | Quantity Unit:Piece",
            "attribute_source": "214 | 214 | 212 | 213",
            "image_url": None,
        },
        {
            "display_id": 200000000001,
            "glcat_mcat_name": "JCB Excavator",
            "eto_ofr_title": "Plastic 4dx JCB Excavator Toy",
            "available_isq_at_approval": 5,
            "eto_ofr_desc": "",
            "eto_ofr_mcat_id": 6032,
            "specifications": "Material:Plastic | Color:Yellow | Quantity:50 | Quantity Unit:Piece",
            "attribute_source": "214 | 214 | 212 | 213",
            "image_url": None,
        },
    ]
    with open(BUYLEAD_JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(sample, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved sample → {BUYLEAD_JSON_FILE}")

# ── ISQ config DB ─────────────────────────────────────────────
MOCK_ISQ_DB = {
    25024: ["Bag Type", "Material", "Color", "Size", "Brand"],
    6032:  ["Bucket Capacity", "Engine Power", "Operating Weight", "Brand", "Condition"],
    1234:  ["Type", "Grade", "Moisture", "Packaging", "Origin"],
}

def get_mandatory_isqs(mcat_id):
    return MOCK_ISQ_DB.get(mcat_id, [])

def get_all_isqs(mcat_id):
    return MOCK_ISQ_DB.get(mcat_id, [])

# ── Run ───────────────────────────────────────────────────────
os.makedirs("/content/drive/MyDrive/buylead_rca/index",   exist_ok=True)
os.makedirs("/content/drive/MyDrive/buylead_rca/outputs", exist_ok=True)

df_mcat = load_mcat_dump(MCAT_FILE)
index, metadata, encoder_model = build_mcat_index(df_mcat)

save_sample_data()
buyLeads = load_buyLeads(BUYLEAD_JSON_FILE)

print(f"\nFirst BuyLead:")
pprint.pprint(buyLeads[0])
print(f"\n✅ Data prep complete. Total BuyLeads: {len(buyLeads)}")

✅ Loaded 98,239 MCATs
⚡ Index already exists — loading from disk


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded index: 98,239 vectors
ℹ️  Already exists — not overwriting.
✅ Loaded 170404 BuyLeads

First BuyLead:
{'attribute_source': '214 | 215 | 214 | 1000 | 1000 | 212 | 213',
 'available_isq_at_approval': 5,
 'display_id': 144097681163,
 'eto_ofr_desc': '',
 'eto_ofr_mcat_id': 25024,
 'eto_ofr_title': 'Cotton Carry Bags',
 'glcat_mcat_name': 'Cloth Bags',
 'image_url': None,
 'specifications': 'Bag Type:Printed | Buyer Filled Details:Buyer needs '
                   'customized design with print. | Material:Cotton | Probable '
                   'Order Value:Rs. 4,500 - 5,200 | Probable Requirement '
                   'Type:Business Use | Quantity:100 | Quantity Unit:Piece'}

✅ Data prep complete. Total BuyLeads: 170404


In [65]:
# %%writefile /content/drive/MyDrive/buylead_rca/02_rule_engine.py

import re
from dataclasses import dataclass
from collections import defaultdict

# ══════════════════════════════════════════════════════════════
# DATABASE CONNECTION
# ══════════════════════════════════════════════════════════════
# Install: !pip install pymysql
# Fill in your actual DB credentials:

DB_CONFIG = {
    "host":     "YOUR_DB_HOST",
    "user":     "YOUR_DB_USER",
    "password": "YOUR_DB_PASSWORD",
    "database": "YOUR_DB_NAME",
    "port":     3306,
}

def get_db_connection():
    import pymysql
    return pymysql.connect(
        host=DB_CONFIG["host"], user=DB_CONFIG["user"],
        password=DB_CONFIG["password"], database=DB_CONFIG["database"],
        port=DB_CONFIG["port"], charset="utf8mb4", connect_timeout=5,
    )

# ══════════════════════════════════════════════════════════════
# ISQ LOOKUP — Real DB query with mock fallback
# ══════════════════════════════════════════════════════════════
# Your query:
#   SELECT im_spec_master_desc AS Q_Desc
#   FROM im_specification_master
#   JOIN im_cat_specification ON im_spec_master_id = fk_im_spec_master_id
#   WHERE im_spec_master_buyer_seller = 1
#     AND IM_CAT_SPEC_STATUS = 1
#     AND im_cat_spec_category_id = mcat_id
#
# Returns: all field names buyer should fill for this MCAT
# e.g. ["Material", "Color", "Size", "Brand", "Bag Type"]

_isq_cache = {}   # cache so we don't hit DB for same MCAT twice

# Mock DB — used when real DB not connected
MOCK_ISQ_DB = {
    25024: ["Bag Type", "Material", "Color", "Size", "Brand"],
    6032:  ["Bucket Capacity", "Engine Power", "Operating Weight", "Brand", "Condition"],
    1234:  ["Type", "Grade", "Moisture", "Packaging", "Origin"],
    16805: ["Bed Type", "Frame Material", "Width", "Brand"],
    3847:  ["Material", "Size", "Usage Type"],
    858:   ["Color", "Material", "Weight", "Brand"],
    28280: ["Material", "Type", "Brand"],
    86:    ["Material", "Capacity", "Size"],
}

def get_mandatory_isqs(mcat_id: int) -> list:
    """
    Returns list of ISQ field names for this MCAT.
    Tries real DB first, falls back to MOCK_ISQ_DB if DB not configured.
    All returned fields = required (buyer should answer all of them).
    """
    if mcat_id in _isq_cache:
        return _isq_cache[mcat_id]

    # Try real DB if credentials are set
    if DB_CONFIG["host"] != "YOUR_DB_HOST":
        try:
            conn   = get_db_connection()
            cursor = conn.cursor()
            cursor.execute("""
                SELECT im_spec_master_desc AS Q_Desc
                FROM   im_specification_master
                JOIN   im_cat_specification
                       ON im_spec_master_id = fk_im_spec_master_id
                WHERE  im_spec_master_buyer_seller = 1
                  AND  IM_CAT_SPEC_STATUS = 1
                  AND  im_cat_spec_category_id = %s
            """, (mcat_id,))
            rows   = cursor.fetchall()
            conn.close()
            fields = [row[0].strip() for row in rows if row[0]]
            _isq_cache[mcat_id] = fields
            return fields
        except Exception as e:
            print(f"⚠️  DB query failed for MCAT {mcat_id}: {e} — using mock")

    # Fallback to mock
    fields = MOCK_ISQ_DB.get(mcat_id, [])
    _isq_cache[mcat_id] = fields
    return fields

def clear_isq_cache():
    global _isq_cache
    _isq_cache = {}

# ══════════════════════════════════════════════════════════════
# ATTRIBUTE SOURCE CLASSIFICATION
# ══════════════════════════════════════════════════════════════
# Source codes tell us WHO filled each ISQ field:
#
#   1–205    → Buyer Filled ISQ        (STRONGEST signal)
#   207–221  → Agent Filled ISQ        (good, human verified)
#   206, 230–260 → Auto Filled by LEAP (system generated)
#   1000     → Predicted by Model      (weakest, often wrong)
#
# Scoring philosophy:
#   Buyer filled  → trust the value fully
#   Agent filled  → trust but flag if contradicts title
#   Auto filled   → lower weight, may be wrong
#   Predicted     → flag if contradicts other fields
# ══════════════════════════════════════════════════════════════

def classify_source(source_code):
    """Returns source type string for a single attribute_source value."""
    try:
        s = int(str(source_code).strip())
    except (ValueError, TypeError):
        return "UNKNOWN"
    if 1 <= s <= 205:
        return "BUYER"
    elif 207 <= s <= 221:
        return "AGENT"
    elif s == 206 or (230 <= s <= 260):
        return "AUTO"
    elif s == 1000:
        return "PREDICTED"
    elif s == 1001:
        return "SYSTEM"
    else:
        return "UNKNOWN"


def parse_specifications(spec_str):
    """
    "Bag Type:Printed | Material:Cotton | Quantity:100"
    → {"Bag Type": "Printed", "Material": "Cotton", "Quantity": "100"}
    """
    result = {}
    if not spec_str or not isinstance(spec_str, str):
        return result
    for pair in spec_str.split("|"):
        pair = pair.strip()
        if ":" in pair:
            key, _, val = pair.partition(":")
            result[key.strip()] = val.strip()
    return result


def parse_attribute_sources(source_str):
    """
    "214 | 215 | 1000 | 212" → [214, 215, 1000, 212]
    """
    if not source_str or not isinstance(source_str, str):
        return []
    try:
        return [int(x.strip()) for x in source_str.split("|")]
    except ValueError:
        return []


def get_field_sources(spec_str, source_str):
    """
    Returns dict mapping each field name to its source type.
    e.g. {"Bag Type": "BUYER", "Material": "BUYER", "Probable Order Value": "PREDICTED"}
    """
    specs   = parse_specifications(spec_str)
    sources = parse_attribute_sources(source_str)
    keys    = list(specs.keys())

    field_sources = {}
    for i, key in enumerate(keys):
        src_code = sources[i] if i < len(sources) else None
        field_sources[key] = {
            "value":       specs[key],
            "source_code": src_code,
            "source_type": classify_source(src_code) if src_code is not None else "UNKNOWN",
        }
    return field_sources


def get_buyer_filled_fields(spec_str, source_str):
    """Returns set of field names filled by buyer (source 1-205)."""
    field_sources = get_field_sources(spec_str, source_str)
    return {k for k, v in field_sources.items() if v["source_type"] == "BUYER"}


# ══════════════════════════════════════════════════════════════
# RCA FLAG DATA STRUCTURE
# ══════════════════════════════════════════════════════════════
@dataclass
class RCAFlag:
    flag_id:          str
    severity:         str    # CRITICAL / HIGH / MEDIUM / LOW
    confidence:       float  # 0.0 – 1.0
    title:            str
    evidence:         str
    suggested_action: str
    score_impact:     float
    source:           str = "rule_engine"


# ══════════════════════════════════════════════════════════════
# WORD LISTS
# ══════════════════════════════════════════════════════════════
GENERIC_TITLES = {
    "product", "item", "goods", "material", "stuff", "thing",
    "object", "unnamed", "untitled", "n/a", "na", "test", "sample",
}

VAGUE_TITLE_WORDS = [
    "and", "or", "parts", "accessories", "items", "products",
    "goods", "materials", "services", "solutions", "systems",
]

B2C_SIGNALS_TITLE = [
    "birthday", "gift", "for me", "personal use", "home use",
    "i need", "buy one", "single piece", "for myself",
]

B2C_SIGNALS_SPEC = [
    "personal use", "ye milegi", "for me", "home use",
    "apne liye", "ghar ke liye", "self use",
]

KEYWORD_STUFFING = [
    "best price", "cheap", "low price", "wholesale rate",
    "best quality", "top quality", "best deal", "no. 1",
    "100% original", "super quality",
]

VAGUE_DESCRIPTIONS = [
    "leading manufacturer", "leading exporter", "leading supplier",
    "we are manufacturer", "we are exporter", "quality products",
    "best quality", "competitive price", "all over india",
    "we provide", "we offer", "we deal", "we supply",
    "contact us", "call us", "for more details",
]

MATERIALS = [
    "plastic", "steel", "iron", "aluminum", "aluminium", "copper",
    "brass", "rubber", "wood", "wooden", "cotton", "polyester",
    "nylon", "glass", "ceramic", "leather", "fabric", "silk",
    "jute", "paper", "stainless steel", "mild steel", "cast iron",
]

CONTACT_PATTERNS = [
    re.compile(r'\b[6-9]\d{9}\b'),
    re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'),
    re.compile(r'\bwhatsapp\b', re.IGNORECASE),
    re.compile(r'\bt\.me\b|\btelegram\b', re.IGNORECASE),
]

SPAM_WORDS = [
    "test", "testing", "asdf", "qwerty", "xxxxx", "aaaa",
    "dummy", "demo", "temp", "fake", "abcd",
]

PROHIBITED_WORDS = [
    "fake currency", "counterfeit", "pirated",
    "narcotics", "drugs", "weapon", "gun", "pistol",
]


# ══════════════════════════════════════════════════════════════
# RULE 1 — TITLE QUALITY
# ══════════════════════════════════════════════════════════════
def check_title_quality(bl):
    flags = []
    title       = (bl.get("eto_ofr_title") or "").strip()
    title_lower = title.lower()
    words       = title_lower.split()

    # 1a. Empty / too short
    if len(title) < 3:
        return [RCAFlag(
            flag_id="EMPTY_TITLE", severity="CRITICAL", confidence=1.0,
            title="Title is empty or too short",
            evidence=f"Title: '{title}' — less than 3 characters.",
            suggested_action="Ask buyer to enter a descriptive product title.",
            score_impact=-40,
        )]

    # 1b. Spam / junk
    for word in SPAM_WORDS:
        if word in title_lower:
            flags.append(RCAFlag(
                flag_id="SPAM_TITLE", severity="CRITICAL", confidence=0.95,
                title="Title appears to be spam or test entry",
                evidence=f"Spam word '{word}' found in title: '{title}'",
                suggested_action="Block BuyLead and route to review queue.",
                score_impact=-50,
            ))
            return flags  # no need to check further

    # 1c. Generic single word
    if title_lower in GENERIC_TITLES:
        flags.append(RCAFlag(
            flag_id="GENERIC_TITLE", severity="CRITICAL", confidence=0.95,
            title="Title is a generic meaningless word",
            evidence=f"Title '{title}' is too generic to identify a product.",
            suggested_action="Replace with specific product name.",
            score_impact=-35,
        ))

    # 1d. Vague multi-word title (e.g. "Building and Flooring Services")
    vague_hits = [w for w in VAGUE_TITLE_WORDS if w in words]
    if len(vague_hits) >= 2 and len(words) <= 4:
        flags.append(RCAFlag(
            flag_id="VAGUE_TITLE", severity="HIGH", confidence=0.80,
            title="Title is too vague to identify a specific product",
            evidence=f"Title '{title}' contains vague words: {vague_hits}. Hard to match to suppliers.",
            suggested_action="Use specific product name e.g. 'Epoxy Floor Coating' not 'Building Services'.",
            score_impact=-15,
        ))

    # 1e. B2C language in B2B marketplace
    b2c_hits = [s for s in B2C_SIGNALS_TITLE if s in title_lower]
    if b2c_hits:
        flags.append(RCAFlag(
            flag_id="B2C_TITLE_LANGUAGE", severity="MEDIUM", confidence=0.85,
            title="Title contains consumer/personal-use language in B2B marketplace",
            evidence=f"Found B2C signals: {b2c_hits} in title '{title}'.",
            suggested_action="Rewrite title to reflect bulk/business purchase intent.",
            score_impact=-10,
        ))

    # 1f. Keyword stuffing
    kw_hits = [s for s in KEYWORD_STUFFING if s in title_lower]
    if len(kw_hits) >= 2:
        flags.append(RCAFlag(
            flag_id="KEYWORD_STUFFED_TITLE", severity="LOW", confidence=0.85,
            title="Title appears keyword-stuffed",
            evidence=f"Found promotional keywords: {kw_hits}",
            suggested_action="Remove promotional words, keep only product name.",
            score_impact=-5,
        ))

    # 1g. Too long (> 15 words)
    if len(words) > 15:
        flags.append(RCAFlag(
            flag_id="TITLE_TOO_LONG", severity="LOW", confidence=0.90,
            title=f"Title is too long ({len(words)} words)",
            evidence=f"Title has {len(words)} words. Ideal: 3-10 words.",
            suggested_action="Shorten title to core product name.",
            score_impact=-5,
        ))

    return flags


# ══════════════════════════════════════════════════════════════
# RULE 2 — DESCRIPTION QUALITY
# ══════════════════════════════════════════════════════════════
def check_description_quality(bl):
    flags = []
    title = (bl.get("eto_ofr_title") or "").strip().lower()
    desc  = (bl.get("eto_ofr_desc")  or "").strip()

    # Empty description is FINE — LLM will auto-generate it
    # Only check quality if description actually exists
    if len(desc) < 5:
        return []   # no flag, no penalty

    desc_lower = desc.lower()

    # 2a. Description identical to title — no value added
    if desc_lower == title:
        flags.append(RCAFlag(
            flag_id="DESCRIPTION_COPY_OF_TITLE", severity="MEDIUM", confidence=1.0,
            title="Description is identical to title — no added value",
            evidence="eto_ofr_desc is a copy of eto_ofr_title.",
            suggested_action="Add specific requirements: quantity, grade, usage, delivery timeline.",
            score_impact=-8,
        ))

    # 2b. Boilerplate / vague seller text
    vague_hits = [p for p in VAGUE_DESCRIPTIONS if p in desc_lower]
    if len(vague_hits) >= 2:
        flags.append(RCAFlag(
            flag_id="VAGUE_DESCRIPTION", severity="MEDIUM", confidence=0.85,
            title="Description is generic boilerplate — not specific to this product",
            evidence=f"Found generic phrases: {vague_hits}",
            suggested_action="Replace with product-specific requirement details.",
            score_impact=-8,
        ))

    # 2c. B2C language in description (like "Personal Use, Ye milegi")
    b2c_hits = [s for s in B2C_SIGNALS_SPEC if s in desc_lower]
    if b2c_hits:
        flags.append(RCAFlag(
            flag_id="B2C_DESCRIPTION", severity="HIGH", confidence=0.90,
            title="Description indicates personal/consumer use — not B2B intent",
            evidence=f"Found consumer-use signals in description: {b2c_hits}. e.g. '{desc[:80]}'",
            suggested_action="Verify this is a genuine B2B bulk purchase requirement.",
            score_impact=-15,
        ))

    # 2d. Contact info in description
    for pattern in CONTACT_PATTERNS:
        match = pattern.search(desc)
        if match:
            flags.append(RCAFlag(
                flag_id="CONTACT_IN_DESCRIPTION", severity="HIGH", confidence=0.95,
                title="Off-platform contact info in description",
                evidence=f"Found '{match.group()}' in description.",
                suggested_action="Remove contact info — violates marketplace policy.",
                score_impact=-15,
            ))
            break

    return flags


# ══════════════════════════════════════════════════════════════
# RULE 3 — ISQ COMPLETENESS WITH SOURCE SCORING
# ══════════════════════════════════════════════════════════════
def check_isq_completeness(bl):
    flags    = []
    mcat_id  = bl.get("eto_ofr_mcat_id")
    spec_str = bl.get("specifications",   "")
    src_str  = bl.get("attribute_source", "")

    # If no ISQs were shown to buyer, skip — cannot penalize for unfilled questions
    # that were never asked
    isq_shown = bl.get("available_isq_at_approval") or 0
    if isq_shown == 0:
        return []

    if not mcat_id or mcat_id == 0:
        return []

    required_fields = get_mandatory_isqs(int(mcat_id))
    if not required_fields:
        return []

    field_sources     = get_field_sources(spec_str, src_str)
    filled_keys_lower = {k.lower() for k in field_sources.keys()}
    required_lower    = {f.lower(): f for f in required_fields}

    missing = [
        required_lower[rf]
        for rf in required_lower
        if rf not in filled_keys_lower
    ]

    if missing:
        filled_count = len(required_fields) - len(missing)
        total_count  = len(required_fields)
        completeness = filled_count / total_count
        penalty      = round((1 - completeness) * 20, 1)

        flags.append(RCAFlag(
            flag_id="MISSING_ISQ_FIELDS",
            severity="HIGH" if completeness < 0.5 else "MEDIUM",
            confidence=1.0,
            title=f"{len(missing)} required ISQ field(s) not filled",
            evidence=(
                f"{isq_shown} ISQ questions were shown to buyer. "
                f"MCAT {mcat_id} requires: {required_fields}. "
                f"Missing: {missing}. "
                f"Completeness: {filled_count}/{total_count}."
            ),
            suggested_action=f"Prompt buyer to fill: {', '.join(missing)}.",
            score_impact=-penalty,
        ))

    return flags


# ══════════════════════════════════════════════════════════════
# RULE 4 — ATTRIBUTE SOURCE QUALITY SCORING
# ══════════════════════════════════════════════════════════════

# These field values should MATCH something in the title
# If they don't → likely a wrong prediction/auto-fill
CHECKABLE_FIELDS = {
    "material", "type", "form", "grade", "inverter type",
    "phase", "bed type", "frame material", "bag type",
    "pipe type", "packaging type", "automation grade",
}

# Generic filler values that are meaningless — always flag
MEANINGLESS_VALUES = {
    "n/a", "na", "not applicable", "other", "any", "none",
    "standard", "normal", "general", "common", "regular",
    "as per requirement", "as required", "as per need",
    "contact us", "to be decided", "tbd", "-", "--", "0",
}


def is_value_relevant_to_title(field_name, field_value, title):
    """
    Checks if an auto-filled or predicted ISQ value makes sense
    for the BuyLead title.

    Logic:
      We check if the value's key words appear in or are consistent
      with the title. This is a lightweight keyword check —
      the NLP pipeline does deeper contradiction detection.

    Examples:
      Title: "Cotton Carry Bags"
        Material=Cotton     → "cotton" in title → RELEVANT ✅
        Material=Polyester  → "polyester" not in title → SUSPICIOUS ⚠️

      Title: "Plastic JCB Excavator Toy"
        Type=Industrial     → "industrial" not in title,
                              "toy" contradicts → SUSPICIOUS ⚠️
        Material=Plastic    → "plastic" in title → RELEVANT ✅

      Title: "K Solare 5 Kw Solar Inverter"
        Inverter Type=On-Grid → "on-grid" or "grid" in title → RELEVANT ✅
        Phase=Single Phase    → reasonable for 5kw → can't verify from title
    """
    title_lower = title.lower()
    value_lower = field_value.lower().strip()

    # Check if value is a meaningless filler
    if value_lower in MEANINGLESS_VALUES:
        return False, "meaningless_value"

    # Check if value words appear in title (direct relevance)
    value_words = set(value_lower.replace("-", " ").split())
    title_words = set(title_lower.replace("-", " ").split())

    # Remove stopwords
    noise = {"and", "or", "the", "of", "for", "in", "a", "an",
             "with", "by", "to", "is", "at", "on"}
    value_words -= noise
    title_words -= noise

    if not value_words:
        return True, "value_too_short_to_check"

    overlap = value_words & title_words
    if overlap:
        return True, f"value words {overlap} found in title"

    # Check for known contradictions between value and title
    CONTRADICTIONS = [
        # (value_keyword, title_keyword_that_conflicts)
        ("industrial",  "toy"),
        ("heavy duty",  "toy"),
        ("commercial",  "personal"),
        ("on-grid",     "off-grid"),
        ("off-grid",    "on-grid"),
        ("single phase","three phase"),
        ("three phase", "single phase"),
        ("automatic",   "manual"),
        ("manual",      "automatic"),
        ("synthetic",   "cotton"),
        ("synthetic",   "natural"),
        ("plastic",     "steel"),
        ("steel",       "plastic"),
        ("wooden",      "steel"),
        ("steel",       "wooden"),
    ]
    for val_kw, title_kw in CONTRADICTIONS:
        if val_kw in value_lower and title_kw in title_lower:
            return False, f"value '{val_kw}' contradicts title word '{title_kw}'"

    # Could not verify either way — treat as neutral
    return True, "could_not_verify_from_title"


def check_attribute_source_quality(bl):
    flags    = []
    spec_str = bl.get("specifications",   "")
    src_str  = bl.get("attribute_source", "")
    title    = (bl.get("eto_ofr_title")   or "").strip()

    # If no ISQs were shown to buyer, skip
    isq_shown = bl.get("available_isq_at_approval") or 0
    if isq_shown == 0:
        return []

    if not spec_str or not src_str:
        return []

    field_sources = get_field_sources(spec_str, src_str)
    if not field_sources:
        return []

    # Exclude meta/system fields from scoring
    exclude_keys = {
        "probable order value", "probable requirement type",
        "buyer filled details", "quantity unit",
    }

    scoreable = {
        k: v for k, v in field_sources.items()
        if k.lower() not in exclude_keys
    }
    if not scoreable:
        return []

    # Count by source type
    counts = defaultdict(int)
    for v in scoreable.values():
        counts[v["source_type"]] += 1

    total    = len(scoreable)
    pred_pct = counts["PREDICTED"] / total

    # ── 4a. Mostly predicted ─────────────────────────────────
    if pred_pct > 0.6 and isq_shown > 0:
        flags.append(RCAFlag(
            flag_id="MOSTLY_PREDICTED_ISQS",
            severity="MEDIUM", confidence=0.85,
            title=f"{int(pred_pct*100)}% of ISQ fields are model predictions — buyer skipped filling",
            evidence=(
                f"{isq_shown} ISQ questions shown to buyer. "
                f"Out of {total} scoreable fields: "
                f"Buyer={counts['BUYER']}, Agent={counts['AGENT']}, "
                f"Auto={counts['AUTO']}, Predicted={counts['PREDICTED']}."
            ),
            suggested_action="Prompt buyer to fill key fields — predictions may be inaccurate.",
            score_impact=-10,
        ))

    # ── 4b. Check auto/predicted values vs title ──────────────
    # This is the new check: are system-generated values meaningful
    # and consistent with what the title says?
    suspicious_fields  = []
    meaningless_fields = []

    for field_name, field_info in field_sources.items():
        src_type    = field_info["source_type"]
        field_value = field_info["value"]
        f_lower     = field_name.lower()

        # Only check auto-filled and predicted values
        # (buyer-filled and agent-filled are trusted)
        if src_type not in ("AUTO", "PREDICTED"):
            continue

        # Only check fields that are meaningful to validate against title
        if f_lower not in CHECKABLE_FIELDS:
            continue

        if not field_value or not field_value.strip():
            continue

        is_relevant, reason = is_value_relevant_to_title(
            field_name, field_value, title
        )

        if not is_relevant:
            if reason == "meaningless_value":
                meaningless_fields.append({
                    "field":  field_name,
                    "value":  field_value,
                    "source": src_type,
                    "reason": reason,
                })
            else:
                suspicious_fields.append({
                    "field":  field_name,
                    "value":  field_value,
                    "source": src_type,
                    "reason": reason,
                })

    # Flag meaningless auto/predicted values
    if meaningless_fields:
        field_list = [f"{f['field']}={f['value']} ({f['source']})"
                      for f in meaningless_fields]
        flags.append(RCAFlag(
            flag_id="MEANINGLESS_AUTO_ISQ_VALUES",
            severity="MEDIUM", confidence=0.95,
            title=f"{len(meaningless_fields)} auto/predicted ISQ field(s) have meaningless values",
            evidence=(
                f"Fields with meaningless values: {field_list}. "
                f"Title: '{title}'. "
                f"These values add no information for supplier matching."
            ),
            suggested_action=(
                "Ask buyer to fill these fields properly, or remove them "
                "from the BuyLead to avoid confusing suppliers."
            ),
            score_impact=-8,
        ))

    # Flag suspicious/contradicting auto/predicted values
    if suspicious_fields:
        field_list = [
            f"{f['field']}={f['value']} ({f['source']}) — {f['reason']}"
            for f in suspicious_fields
        ]
        flags.append(RCAFlag(
            flag_id="SUSPICIOUS_AUTO_ISQ_VALUES",
            severity="HIGH", confidence=0.80,
            title=f"{len(suspicious_fields)} auto/predicted ISQ value(s) inconsistent with title",
            evidence=(
                f"Title: '{title}'. "
                f"Suspicious auto/predicted values: {field_list}. "
                f"These values may be wrong predictions that don't match the product."
            ),
            suggested_action=(
                "Verify these auto-filled/predicted values are correct for this product. "
                "Wrong predictions mislead supplier matching."
            ),
            score_impact=-12,
        ))

    # ── 4c. Buyer vs predicted contradiction ──────────────────
    buyer_filled_values = {
        k.lower(): v["value"].lower()
        for k, v in field_sources.items()
        if v["source_type"] == "BUYER"
    }
    predicted_values = {
        k.lower(): v["value"].lower()
        for k, v in field_sources.items()
        if v["source_type"] == "PREDICTED"
    }

    req_type_key = "probable requirement type"
    if req_type_key in predicted_values:
        predicted_req  = predicted_values[req_type_key]
        all_buyer_text = " ".join(buyer_filled_values.values())
        b2c_in_buyer   = [s for s in B2C_SIGNALS_SPEC if s in all_buyer_text]

        if b2c_in_buyer and "business" in predicted_req:
            flags.append(RCAFlag(
                flag_id="BUYER_B2C_VS_PREDICTED_B2B",
                severity="HIGH", confidence=0.88,
                title="Buyer filled personal-use details but system predicted Business Use",
                evidence=(
                    f"Buyer filled fields contain: {b2c_in_buyer}. "
                    f"But 'Probable Requirement Type' (predicted) = '{predicted_req}'. "
                    f"This contradicts buyer's actual intent."
                ),
                suggested_action=(
                    "Verify buyer intent — likely a personal/consumer query, not B2B. "
                    "Consider removing from B2B supplier routing."
                ),
                score_impact=-20,
            ))

    # ── 4d. Zero buyer-filled fields ─────────────────────────
    if counts["BUYER"] == 0 and total > 0 and isq_shown > 0:
        flags.append(RCAFlag(
            flag_id="NO_BUYER_FILLED_ISQS",
            severity="HIGH",    # HIGH — buyer was shown questions and filled none
            confidence=0.95,
            title=f"No ISQ fields filled by buyer despite {isq_shown} questions shown",
            evidence=(
                f"{isq_shown} ISQ questions were shown to buyer at approval time. "
                f"All {total} spec fields were filled by system (auto/predicted). "
                f"Buyer engagement = 0%. This significantly reduces BuyLead reliability."
            ),
            suggested_action=(
                f"Re-engage buyer to fill at least key ISQ fields. "
                f"Auto/predicted values for {total} fields may be inaccurate."
            ),
            score_impact=-15,
        ))

    return flags


# ══════════════════════════════════════════════════════════════
# RULE 5 — SPECIFICATION-TITLE MISMATCH
# ══════════════════════════════════════════════════════════════
def check_spec_title_mismatch(bl):
    flags     = []
    title     = (bl.get("eto_ofr_title") or "").strip().lower()
    spec_str  = bl.get("specifications", "")
    specs     = parse_specifications(spec_str)

    # 5a. Material contradiction
    title_materials = [m for m in MATERIALS if m in title]
    spec_material   = specs.get("Material", "").lower()
    if title_materials and spec_material:
        if not any(m in spec_material for m in title_materials):
            flags.append(RCAFlag(
                flag_id="MATERIAL_MISMATCH",
                severity="HIGH", confidence=0.85,
                title="Material in title contradicts Material in specifications",
                evidence=(
                    f"Title implies material: {title_materials}. "
                    f"Spec 'Material' says: '{spec_material}'."
                ),
                suggested_action=f"Correct either title or Material spec — they contradict each other.",
                score_impact=-12,
            ))

    # 5b. B2C signals in specifications (personal use, ye milegi etc.)
    all_spec_text = " ".join(specs.values()).lower()
    b2c_spec_hits = [s for s in B2C_SIGNALS_SPEC if s in all_spec_text]
    if b2c_spec_hits:
        flags.append(RCAFlag(
            flag_id="B2C_SIGNALS_IN_SPECS",
            severity="HIGH", confidence=0.90,
            title="Specifications contain personal/consumer-use language",
            evidence=(
                f"Found B2C signals in specs: {b2c_spec_hits}. "
                f"Spec text: '{all_spec_text[:150]}'"
            ),
            suggested_action="Verify buyer intent — this looks like a personal query, not B2B bulk purchase.",
            score_impact=-18,
        ))

    # 5c. NOTE: MCAT semantic correctness is checked by 03_mcat_ranker.py
    # using embeddings + cross-encoder which understands meaning.
    # e.g. "E Rickshaw Loader Flooring Mat" → "Automotive Floor Mats" ✅
    #      "Aata Chakki" → "Flour Mill" ✅
    # Word overlap here would cause false positives for such valid mappings.
    # Rule engine only catches MCAT_UNMAPPED (Rule 9) — ranker does the rest.

    return flags


# ══════════════════════════════════════════════════════════════
# RULE 6 — QUANTITY SANITY
# ══════════════════════════════════════════════════════════════
def check_quantity_sanity(bl):
    """
    Only flags quantities that are absurd/unrealistic for the product.
    Quantity=1 is valid — buyer decides how many they need.
    """
    specs    = parse_specifications(bl.get("specifications", ""))
    qty_str  = specs.get("Quantity", "").replace(",", "").strip()
    title    = (bl.get("eto_ofr_title") or "").lower()
    qty_unit = specs.get("Quantity Unit", "").lower()

    if not qty_str:
        return []
    try:
        qty = float(qty_str)
    except ValueError:
        return []

    # a. Zero or negative
    if qty <= 0:
        return [RCAFlag(
            flag_id="ZERO_QUANTITY", severity="HIGH", confidence=1.0,
            title="Quantity is zero or negative",
            evidence=f"Quantity: {qty}. A valid positive quantity is required.",
            suggested_action="Ask buyer to enter a valid quantity.",
            score_impact=-8,
        )]

    # b. Absurdly high (>10 million for any product)
    if qty > 10_000_000:
        return [RCAFlag(
            flag_id="UNREALISTIC_QUANTITY", severity="HIGH", confidence=0.92,
            title="Quantity unrealistically high — likely extra zeros typo",
            evidence=f"Quantity: {qty:,.0f} {qty_unit}. Almost certainly a typo.",
            suggested_action=f"Confirm with buyer: did you mean {qty/1000:,.0f} or {qty/100:,.0f}?",
            score_impact=-8,
        )]

    # c. High quantity for heavy/expensive single-unit products
    SINGLE_UNIT_KEYWORDS = [
        "excavator", "crane", "bulldozer", "tractor", "generator",
        "transformer", "turbine", "boiler", "furnace", "reactor",
        "mri", "ct scan", "x-ray machine", "lathe machine", "cnc machine",
        "car", "truck", "bus", "ambulance",
    ]
    if any(kw in title for kw in SINGLE_UNIT_KEYWORDS) and qty > 500:
        return [RCAFlag(
            flag_id="HIGH_QTY_FOR_HEAVY_PRODUCT", severity="MEDIUM", confidence=0.80,
            title=f"Quantity {qty:,.0f} seems unrealistic for this product",
            evidence=(
                f"Title suggests heavy/capital equipment. "
                f"Quantity {qty:,.0f} {qty_unit} is unusually high for this category."
            ),
            suggested_action="Verify quantity with buyer — may be a data entry error.",
            score_impact=-6,
        )]

    # d. Bulk commodity — absurdly large tonnage
    if qty_unit in ("ton", "tonne", "mt") and qty > 100_000:
        return [RCAFlag(
            flag_id="UNREALISTIC_BULK_QUANTITY", severity="HIGH", confidence=0.88,
            title=f"Quantity {qty:,.0f} {qty_unit} is unrealistically large",
            evidence=f"Even large commodity orders rarely exceed 10,000 tons.",
            suggested_action=f"Confirm with buyer: did you mean {qty/1000:,.0f} tons?",
            score_impact=-8,
        )]

    return []



# ══════════════════════════════════════════════════════════════
# RULE 7 — PROHIBITED / SPAM CONTENT
# ══════════════════════════════════════════════════════════════
def check_prohibited_content(bl):
    title    = (bl.get("eto_ofr_title") or "").lower()
    desc     = (bl.get("eto_ofr_desc")  or "").lower()
    combined = f"{title} {desc}"

    for phrase in PROHIBITED_WORDS:
        if phrase in combined:
            return [RCAFlag(
                flag_id="PROHIBITED_CONTENT", severity="CRITICAL", confidence=0.85,
                title="Potentially prohibited content detected",
                evidence=f"Found '{phrase}' in title or description.",
                suggested_action="Escalate to compliance review. Do not publish.",
                score_impact=-100,
            )]
    return []


# ══════════════════════════════════════════════════════════════
# RULE 8 — CONTACT INFO IN TITLE
# ══════════════════════════════════════════════════════════════
def check_contact_in_title(bl):
    title = bl.get("eto_ofr_title", "") or ""
    for pattern in CONTACT_PATTERNS:
        match = pattern.search(title)
        if match:
            return [RCAFlag(
                flag_id="CONTACT_IN_TITLE", severity="HIGH", confidence=0.95,
                title="Contact info found in title",
                evidence=f"Found '{match.group()}' in title.",
                suggested_action="Remove contact info from title.",
                score_impact=-15,
            )]
    return []


# ══════════════════════════════════════════════════════════════
# RULE 9 — MCAT IS NULL / UNMAPPED
# ══════════════════════════════════════════════════════════════
def check_mcat_unmapped(bl):
    mcat_id   = bl.get("eto_ofr_mcat_id",   0)
    mcat_name = bl.get("glcat_mcat_name",   "")

    if not mcat_id or mcat_id == 0:
        return [RCAFlag(
            flag_id="MCAT_UNMAPPED", severity="CRITICAL", confidence=1.0,
            title="BuyLead has no category mapped (MCAT ID = 0)",
            evidence="eto_ofr_mcat_id is 0 or null — BuyLead cannot be routed to suppliers.",
            suggested_action="Run MCAT ranker to suggest correct category.",
            score_impact=-30,
        )]

    if not mcat_name or mcat_name.strip() == "":
        return [RCAFlag(
            flag_id="MCAT_NAME_MISSING", severity="HIGH", confidence=1.0,
            title="MCAT ID exists but MCAT name is missing",
            evidence=f"eto_ofr_mcat_id={mcat_id} but glcat_mcat_name is null/empty.",
            suggested_action="Look up MCAT name from MCAT dump using the ID.",
            score_impact=-10,
        )]

    return []


# ══════════════════════════════════════════════════════════════
# MAIN RULE ENGINE
# ══════════════════════════════════════════════════════════════
ALL_RULES = [
    check_prohibited_content,      # check first — exit early if spam
    check_title_quality,
    check_description_quality,
    check_mcat_unmapped,
    check_isq_completeness,
    check_attribute_source_quality,
    check_spec_title_mismatch,
    check_quantity_sanity,
    check_contact_in_title,
]


def run_rule_engine(bl):
    """
    Runs all rule checks on a single BuyLead.
    Returns list of RCAFlag objects.
    """
    flags = []
    for rule_fn in ALL_RULES:
        try:
            result = rule_fn(bl)
            if result:
                flags.extend(result)
        except Exception as e:
            print(f"⚠️  Rule {rule_fn.__name__} failed: {e}")
    return flags


# ══════════════════════════════════════════════════════════════
# TEST — runs automatically when cell executes
# ══════════════════════════════════════════════════════════════
from collections import Counter

# Guard: buyLeads is defined in 01_data_prep cell
# If not found, use a minimal inline test set
try:
    _test_data = buyLeads
    print(f"Running rule engine on {len(_test_data)} BuyLeads from 01_data_prep...\n")
except NameError:
    print("⚠️  buyLeads not found — run 01_data_prep cell first.")
    print("   Using 2 inline test BuyLeads for now.\n")
    _test_data = [
        {
            "display_id": 1,
            "eto_ofr_title": "Cotton Carry Bags",
            "eto_ofr_desc": "",
            "eto_ofr_mcat_id": 25024,
            "glcat_mcat_name": "Cloth Bags",
            "specifications": "Bag Type:Printed | Material:Cotton | Quantity:100 | Quantity Unit:Piece",
            "attribute_source": "214 | 214 | 212 | 213",
            "image_url": None,
        },
        {
            "display_id": 2,
            "eto_ofr_title": "Plastic 4dx JCB Excavator Toy",
            "eto_ofr_desc": "",
            "eto_ofr_mcat_id": 6032,
            "glcat_mcat_name": "JCB Excavator",
            "specifications": "Material:Plastic | Color:Yellow | Quantity:50 | Buyer Filled Details:Personal Use, ye milegi",
            "attribute_source": "214 | 214 | 212 | 230",
            "image_url": None,
        },
    ]

all_results = []
for bl in _test_data:
    flags = run_rule_engine(bl)
    all_results.append({
        "id":    bl["display_id"],
        "title": bl["eto_ofr_title"],
        "mcat":  bl.get("glcat_mcat_name", ""),
        "flags": flags,
    })

# Print per-BuyLead results
icons = {"CRITICAL":"🔴","HIGH":"🟠","MEDIUM":"🟡","LOW":"🔵"}
for r in all_results:
    if r["flags"]:
        print(f"\n{'='*60}")
        print(f"📋 ID: {r['id']}")
        print(f"   Title: {r['title'][:55]}")
        print(f"   MCAT:  {r['mcat'][:40]}")
        for f in r["flags"]:
            print(f"  {icons.get(f.severity,'⚪')} [{f.severity}] {f.flag_id}")
            print(f"     {f.title}")
            print(f"     Evidence: {f.evidence[:120]}")
            print(f"     → {f.suggested_action}")
    else:
        print(f"\n✅ {r['id']} — {r['title'][:50]} — No issues")

# Summary stats
all_flag_ids = [f.flag_id for r in all_results for f in r["flags"]]
print(f"\n{'='*60}")
print(f"📊 TOTAL: {len(all_flag_ids)} flags across {len(buyLeads)} BuyLeads")
print(f"\nFlag frequency:")
for flag_id, count in Counter(all_flag_ids).most_common():
    bar = "█" * count
    print(f"  {flag_id:<35} {count:>3}  {bar}")

Streaming output truncated to the last 5000 lines.

✅ 144280570767 — Flower Pots — No issues

✅ 144280574375 — Car Bass Tube — No issues

📋 ID: 144280574946
   Title: Pradesh Test On
   MCAT:  Cloud Testing Service
  🔴 [CRITICAL] SPAM_TITLE
     Title appears to be spam or test entry
     Evidence: Spam word 'test' found in title: 'Pradesh Test On'
     → Block BuyLead and route to review queue.

📋 ID: 144280575146
   Title: Isopure Whey Protein
   MCAT:  Isopure Whey Protein
  🟠 [HIGH] NO_BUYER_FILLED_ISQS
     No ISQ fields filled by buyer despite 5 questions shown
     Evidence: 5 ISQ questions were shown to buyer at approval time. All 3 spec fields were filled by system (auto/predicted). Buyer en
     → Re-engage buyer to fill at least key ISQ fields. Auto/predicted values for 3 fields may be inaccurate.

✅ 144280584559 — FMC Insecticides — No issues

✅ 144280586075 — Neemalo Face Wash — No issues

📋 ID: 144280587963
   Title: 5 Kg Non Domestic LPG Cylinders
   MCAT:  Commercial LP

In [66]:
# %%writefile /content/drive/MyDrive/buylead_rca/03_mcat_ranker.py

import numpy as np
import faiss
import pickle
import os
import json
import openai
from sentence_transformers import SentenceTransformer, CrossEncoder

# ══════════════════════════════════════════════════════════════
# MCAT RANKER — checks if MCAT mapping is correct
#
# Uses 2 signals + LLM fallback via IndiaMART Gateway:
#
# SIGNAL 1 — TITLE vs MCAT (embedding search)
# SIGNAL 2 — BUYER-FILLED SPECS vs MCAT
# LLM FALLBACK — when embedding score < 0.25 (brand names etc.)
#
# RUN ORDER:
#   01_data_prep   → INDEX_FILE, META_FILE, parse_specifications,
#                    get_field_sources
#   02_rule_engine → RCAFlag
#   This cell      → MCAT ranker
# ══════════════════════════════════════════════════════════════

BI_ENCODER_MODEL    = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K_RETRIEVE      = 20
TOP_K_RETURN        = 3
MISMATCH_SCORE_GAP  = 0.15
UNCERTAIN_SCORE     = 0.25   # below this → LLM verification triggered
MISMATCH_MIN_CONF   = 0.60   # below this → don't flag mismatch


# ══════════════════════════════════════════════════════════════
# LLM GATEWAY — IndiaMART internal gateway
# ══════════════════════════════════════════════════════════════
LLM_GATEWAY_URL   = "https://imllm.intermesh.net/v1"
LLM_GATEWAY_MODEL = "anthropic/claude-sonnet-4-6"
LLM_API_KEY       = os.environ.get("LLM_API_KEY", "")


def get_llm_client():
    """
    Returns OpenAI-compatible client for IndiaMART LLM Gateway.
    Key must be set: os.environ['LLM_API_KEY'] = 'sk-xxx'
    """
    if not LLM_API_KEY:
        print("⚠️  LLM_API_KEY not set — LLM verification disabled")
        print("   Set it with: import os; os.environ['LLM_API_KEY'] = 'sk-xxx'")
        return None
    return openai.OpenAI(
        api_key  = LLM_API_KEY,
        base_url = LLM_GATEWAY_URL,
    )


# ══════════════════════════════════════════════════════════════
# LLM MCAT VERIFICATION
# Called when embedding score < 0.25 (brand name / ambiguous title)
# ══════════════════════════════════════════════════════════════

MCAT_VERIFY_PROMPT = """
You are an expert product cataloger for IndiaMART, India's largest B2B marketplace.

A buyer submitted a BuyLead. Your job: decide if the mapped product category is correct.

BuyLead Title    : {title}
Mapped Category  : {mcat_name}
Buyer filled specs: {specs}

Think step by step:
- Could this title be a brand name? (e.g. "Tiger King" = male enhancement capsule)
- Could this be a Hindi/regional language product name? (e.g. "Aata Chakki" = Flour Mill)
- Does the mapped category make sense for this product?
- Are the buyer specs consistent with the mapped category?

Return ONLY valid JSON (no extra text, no markdown):
{{
  "mapping_correct": true or false,
  "confidence": 0.0 to 1.0,
  "reasoning": "one clear sentence explaining your decision",
  "suggested_category": "better category name if mapping is wrong, else null"
}}
"""


def verify_mcat_with_llm(bl, mapped_mcat_name, llm_client):
    """
    Calls IndiaMART LLM Gateway to verify MCAT mapping.
    Used only when embedding ranker is uncertain (score < 0.25).

    Examples:
      "Tiger King Xxl African Size Capsule" → "Penis Enlargement Product" ✅
      "Aata Chakki Single Phase"            → "Flour Mill"               ✅
      "bolero engine head"                  → "Car Cylinder Head"        ✅
    """
    if not llm_client:
        return None

    title    = (bl.get("eto_ofr_title")  or "").strip()
    spec_str = bl.get("specifications",   "")
    src_str  = bl.get("attribute_source", "")

    # Use buyer-filled specs as context
    try:
        fs = get_field_sources(spec_str, src_str)
        buyer_specs = {
            k: v["value"] for k, v in fs.items()
            if v["source_type"] in ("BUYER", "AGENT")
            and k.lower() not in {
                "quantity", "quantity unit",
                "probable order value", "probable requirement type",
            }
        }
    except Exception:
        buyer_specs = {}

    if not buyer_specs:
        buyer_specs = dict(list(parse_specifications(spec_str).items())[:5])

    prompt = MCAT_VERIFY_PROMPT.format(
        title     = title,
        mcat_name = mapped_mcat_name,
        specs     = str(buyer_specs) if buyer_specs else "none provided",
    )

    fallback = {
        "mapping_correct":    True,
        "confidence":         0.50,
        "reasoning":          "LLM could not verify — defaulting to trust existing mapping",
        "suggested_category": None,
    }

    try:
        response = llm_client.chat.completions.create(
            model    = LLM_GATEWAY_MODEL,
            messages = [
                {
                    "role":    "system",
                    "content": (
                        "You are a product cataloging expert for IndiaMART B2B marketplace. "
                        "Always respond with valid JSON only. No markdown, no explanation outside JSON."
                    ),
                },
                {
                    "role":    "user",
                    "content": prompt,
                },
            ],
            max_tokens  = 300,
            temperature = 0.1,
        )

        text = response.choices[0].message.content.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        text   = text.strip()
        result = json.loads(text)

        if "mapping_correct" not in result:
            print(f"   ⚠️  LLM response missing 'mapping_correct' key")
            return fallback

        return result

    except json.JSONDecodeError as e:
        print(f"   ⚠️  LLM JSON parse error: {e}")
        return fallback
    except Exception as e:
        print(f"   ⚠️  LLM Gateway error: {e}")
        return fallback


# ══════════════════════════════════════════════════════════════
# LOAD MODELS
# ══════════════════════════════════════════════════════════════
def load_ranker_models():
    print("🔄 Loading bi-encoder...")
    bi_encoder    = SentenceTransformer(BI_ENCODER_MODEL)
    print("🔄 Loading cross-encoder...")
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
    print("🔄 Loading FAISS index...")
    index         = faiss.read_index(INDEX_FILE)
    with open(META_FILE, "rb") as f:
        metadata  = pickle.load(f)
    print(f"✅ Ranker ready — {index.ntotal:,} MCATs indexed")
    return bi_encoder, cross_encoder, index, metadata


# ══════════════════════════════════════════════════════════════
# QUERY BUILDER
# ══════════════════════════════════════════════════════════════
QUERY_EXCLUDE_KEYS = {
    "quantity", "quantity unit", "probable order value",
    "probable requirement type", "buyer filled details",
}

GENERIC_SPEC_VALUES = {
    "plastic", "steel", "iron", "other", "standard", "normal",
    "general", "common", "as required", "n/a", "na", "yes", "no",
    "new", "used", "manual", "automatic",
}


def is_spec_value_specific(value):
    v = value.lower().strip()
    if not v or len(v) <= 1:
        return False
    if v in GENERIC_SPEC_VALUES:
        return False
    if v.replace(".", "").replace(",", "").replace(" ", "").isdigit():
        return False
    if len(v) > 60:
        return False
    return True


def build_query(bl):
    """
    Title is PRIMARY signal. Specs added only if specific and not in title.
    Fixes: "Material:Plastic" was dominating search → now excluded if generic.
    """
    title    = (bl.get("eto_ofr_title") or "").strip()
    desc     = (bl.get("eto_ofr_desc")  or "").strip()
    spec_str = bl.get("specifications",   "")
    src_str  = bl.get("attribute_source", "")

    specs         = parse_specifications(spec_str)
    field_sources = get_field_sources(spec_str, src_str)
    title_lower   = title.lower()
    title_words   = set(title_lower.replace("-", " ").split())

    selected = []
    for k, v in specs.items():
        if k.lower() in QUERY_EXCLUDE_KEYS:
            continue
        if not is_spec_value_specific(v):
            continue
        value_words = set(v.lower().replace("-", " ").split()) - {"and", "or", "the", "of"}
        if value_words and value_words.issubset(title_words):
            continue
        src_type = field_sources.get(k, {}).get("source_type", "UNKNOWN")
        priority = 0 if src_type in ("BUYER", "AGENT") else 1
        selected.append((priority, v))

    selected.sort(key=lambda x: x[0])
    query = title
    for _, v in selected[:3]:
        query += f" {v}"

    if len(title.split()) <= 2 and desc:
        query += " " + desc[:100]

    return query.strip()


# ══════════════════════════════════════════════════════════════
# SIGNAL EVALUATORS
# ══════════════════════════════════════════════════════════════
def retrieve_top_k(query, bi_encoder, index, metadata, k=TOP_K_RETRIEVE):
    query_vec = bi_encoder.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    scores, indices = index.search(query_vec, k)
    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        candidates.append({
            "mcat_id":   metadata["mcat_ids"][idx],
            "mcat_name": metadata["mcat_names"][idx],
            "full_path": metadata["full_paths"][idx],
            "cat_name":  metadata["cat_names"][idx],
            "bi_score":  round(float(score), 4),
        })
    return candidates


def rerank_candidates(query, candidates, cross_encoder):
    pairs     = [(query, c["full_path"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)
    sigmoid   = lambda x: 1 / (1 + np.exp(-x))
    ce_norm   = sigmoid(np.array([float(s) for s in ce_scores]))
    for c, cn, cs in zip(candidates, ce_norm, ce_scores):
        c["ce_score"]    = float(cs)
        c["final_score"] = round(0.4 * c["bi_score"] + 0.6 * float(cn), 4)
    candidates.sort(key=lambda x: x["final_score"], reverse=True)
    return candidates


def compute_confidence(candidates):
    if len(candidates) < 2:
        return candidates[0]["final_score"] if candidates else 0.0
    margin = candidates[0]["final_score"] - candidates[1]["final_score"]
    return round(float(1 / (1 + np.exp(-10 * (margin - 0.1)))), 3)


def evaluate_title_signal(candidates, mapped_mcat_id, mapped_mcat_name):
    top1           = candidates[0]
    top1_score     = top1["final_score"]
    top1_different = (top1["mcat_id"] != mapped_mcat_id)

    if top1_score < UNCERTAIN_SCORE:
        return False, 0.0, (
            f"Ranker score {top1_score:.3f} too low — brand name or domain-specific title. "
            f"Trusting existing mapping '{mapped_mcat_name}'."
        )

    mapped_rank = mapped_score = None
    for i, c in enumerate(candidates):
        if c["mcat_id"] == mapped_mcat_id:
            mapped_rank  = i + 1
            mapped_score = c["final_score"]
            break

    if not top1_different:
        return False, 0.0, f"Title matches '{mapped_mcat_name}' (rank #1, score={top1_score:.3f})"

    if mapped_rank is None and top1_score > 0.40:
        return True, 0.85, (
            f"'{mapped_mcat_name}' not in top-20. "
            f"Confident match: '{top1['mcat_name']}' (score={top1_score:.3f})"
        )

    if mapped_rank and mapped_rank > 5 and top1_score > 0.35:
        return True, 0.80, (
            f"'{mapped_mcat_name}' ranks #{mapped_rank} (score={mapped_score:.3f}). "
            f"Better: '{top1['mcat_name']}' at #1 (score={top1_score:.3f})"
        )

    if mapped_score and (top1_score - mapped_score) > MISMATCH_SCORE_GAP and top1_score > 0.35:
        return True, 0.75, (
            f"'{top1['mcat_name']}' scores {top1_score:.3f} vs "
            f"'{mapped_mcat_name}' scores {mapped_score:.3f}"
        )

    return False, 0.0, f"Title close to mapped MCAT (rank #{mapped_rank})"


def evaluate_spec_signal(bl, mapped_mcat_id, mapped_mcat_name, top1):
    field_sources = get_field_sources(
        bl.get("specifications", ""),
        bl.get("attribute_source", "")
    )
    buyer_values = {
        k: v["value"]
        for k, v in field_sources.items()
        if v["source_type"] in ("BUYER", "AGENT")
        and k.lower() not in QUERY_EXCLUDE_KEYS
        and len(v["value"]) < 80
    }

    if not buyer_values:
        return False, 0.0, "No buyer-filled spec values to check"

    spec_text         = " ".join(f"{k} {v}" for k, v in buyer_values.items()).lower()
    mapped_mcat_lower = mapped_mcat_name.lower()

    INCONSISTENCY_PATTERNS = [
        ("plastic",   "excavator"),   ("toy",      "excavator"),
        ("toy",       "machinery"),   ("color",    "excavator"),
        ("cream",     "machinery"),   ("capsule",  "machinery"),
        ("tablet",    "machinery"),   ("cotton",   "excavator"),
        ("fabric",    "machinery"),   ("rice",     "machinery"),
    ]

    conflicts = [
        f"spec has '{sk}' but MCAT is '{mk}'"
        for sk, mk in INCONSISTENCY_PATTERNS
        if sk in spec_text and mk in mapped_mcat_lower
    ]

    if conflicts:
        return True, 0.85, (
            f"Buyer specs contradict MCAT. Conflicts: {conflicts}. "
            f"Specs: {buyer_values}"
        )

    noise        = {"and","or","the","of","for","in","a","an","type","grade","size"}
    spec_words   = set(spec_text.split()) - noise
    mapped_words = set(mapped_mcat_lower.split()) - noise
    top1_words   = set(top1["full_path"].lower().split()) - noise

    if (len(spec_words & top1_words) > len(spec_words & mapped_words) + 1
            and top1["mcat_id"] != mapped_mcat_id):
        return True, 0.70, (
            f"Buyer specs overlap more with '{top1['mcat_name']}' "
            f"than '{mapped_mcat_name}'"
        )

    return False, 0.0, f"Buyer specs consistent with '{mapped_mcat_name}'"


# ══════════════════════════════════════════════════════════════
# MAIN RANKER
# ══════════════════════════════════════════════════════════════
def run_mcat_ranker(bl, bi_encoder, cross_encoder, index, metadata):
    """
    Checks MCAT mapping correctness.
    LLM Gateway called automatically when embedding is uncertain.
    No gemini_model parameter needed — uses LLM_API_KEY from environment.
    """
    mapped_mcat_id   = bl.get("eto_ofr_mcat_id")
    mapped_mcat_name = bl.get("glcat_mcat_name", "") or ""
    query            = build_query(bl)

    # Signal 1: Title embedding search
    candidates = retrieve_top_k(query, bi_encoder, index, metadata)
    if not candidates:
        return [], None

    candidates      = rerank_candidates(query, candidates, cross_encoder)
    top_suggestions = candidates[:TOP_K_RETURN]
    top1            = candidates[0]

    title_fires, title_conf, title_reason = evaluate_title_signal(
        candidates, mapped_mcat_id, mapped_mcat_name
    )

    # Signal 2: Buyer-filled specs
    spec_fires, spec_conf, spec_reason = evaluate_spec_signal(
        bl, mapped_mcat_id, mapped_mcat_name, top1
    )

    # LLM fallback when embedding uncertain
    if top1["final_score"] < UNCERTAIN_SCORE:
        llm_client = get_llm_client()
        if llm_client:
            print(f"   🤖 Ranker uncertain (score={top1['final_score']:.3f}) "
                  f"— asking LLM Gateway ({LLM_GATEWAY_MODEL})...")
            llm_result = verify_mcat_with_llm(bl, mapped_mcat_name, llm_client)
            if llm_result:
                print(f"   🤖 LLM: correct={llm_result['mapping_correct']}, "
                      f"confidence={llm_result['confidence']:.0%}, "
                      f"reason={llm_result['reasoning']}")
                if llm_result["mapping_correct"] and llm_result["confidence"] >= 0.60:
                    title_fires = False
                    spec_fires  = False
                    title_reason = (
                        f"LLM Gateway verified mapping correct: "
                        f"{llm_result['reasoning']} "
                        f"(confidence {llm_result['confidence']:.0%})"
                    )
                elif not llm_result["mapping_correct"] and llm_result["confidence"] >= 0.70:
                    title_fires  = True
                    title_conf   = llm_result["confidence"]
                    title_reason = (
                        f"LLM Gateway says mapping wrong: {llm_result['reasoning']}. "
                        f"Suggested: {llm_result.get('suggested_category', 'unknown')}"
                    )
        else:
            title_fires  = False
            spec_fires   = False
            title_reason = (
                f"Ranker score {top1['final_score']:.3f} too low. "
                f"Set LLM_API_KEY for verification."
            )

    # Combine signals
    active = []
    if title_fires:
        active.append(("TITLE", title_conf, title_reason))
    if spec_fires:
        active.append(("BUYER_SPECS", spec_conf, spec_reason))

    if not active:
        return top_suggestions, None

    combined_conf = round(
        min(0.97, max(s[1] for s in active) + (0.10 if len(active) == 2 else 0.0)),
        3
    )
    signals_label = " + ".join(s[0] for s in active)

    if combined_conf < MISMATCH_MIN_CONF:
        print(f"   ℹ️  Confidence {combined_conf:.0%} below threshold — skipping flag")
        return top_suggestions, None

    mismatch_flag = RCAFlag(
        flag_id="MCAT_MISMATCH",
        severity="CRITICAL" if combined_conf > 0.80 else "HIGH",
        confidence=combined_conf,
        title=(
            f"MCAT '{mapped_mcat_name}' appears incorrect — "
            f"likely '{top1['mcat_name']}' based on {signals_label}"
        ),
        evidence=(
            f"[TITLE] {title_reason} | "
            f"[SPECS] {spec_reason} | "
            f"Suggested: '{top1['full_path']}' (id={top1['mcat_id']})"
        )[:600],
        suggested_action=(
            f"Remap to MCAT {top1['mcat_id']}: '{top1['mcat_name']}' "
            f"(confidence {combined_conf:.0%}). "
            f"Path: {top1['full_path']}"
        ),
        score_impact=-35 if combined_conf > 0.80 else -20,
        source="mcat_ranker",
    )

    return top_suggestions, mismatch_flag


# ══════════════════════════════════════════════════════════════
# TEST
# ══════════════════════════════════════════════════════════════
import os as _os

if not _os.path.exists(INDEX_FILE):
    print(f"⚠️  FAISS index not found: {INDEX_FILE}")
    print("   Run 01_data_prep first.")
    bi_encoder = cross_encoder = faiss_index = mcat_metadata = None
else:
    print("🔄 Loading ranker models...")
    bi_encoder, cross_encoder, faiss_index, mcat_metadata = load_ranker_models()

    # Show LLM status
    if LLM_API_KEY:
        print(f"✅ LLM Gateway ready: {LLM_GATEWAY_MODEL}")
    else:
        print("ℹ️  LLM Gateway disabled — set LLM_API_KEY for brand name verification")

    try:
        _test_data = buyLeads[:5]
        print(f"\n✅ Testing on first 5 real BuyLeads...\n")
    except NameError:
        print("\n⚠️  Using inline test data (run 01_data_prep first)\n")
        _test_data = [
            {
                "display_id": 1,
                "eto_ofr_title": "Plastic 4dx JCB Excavator Toy",
                "eto_ofr_desc": "", "eto_ofr_mcat_id": 6032,
                "glcat_mcat_name": "JCB Excavator",
                "specifications": "Material:Plastic | Color:Yellow | Quantity:50 | Quantity Unit:Piece",
                "attribute_source": "214 | 214 | 212 | 213",
            },
            {
                "display_id": 2,
                "eto_ofr_title": "Cotton Carry Bags",
                "eto_ofr_desc": "", "eto_ofr_mcat_id": 25024,
                "glcat_mcat_name": "Cloth Bags",
                "specifications": "Bag Type:Printed | Material:Cotton | Quantity:100 | Quantity Unit:Piece",
                "attribute_source": "214 | 214 | 212 | 213",
            },
            {
                "display_id": 3,
                "eto_ofr_title": "Tiger King Xxl African Size Capsule",
                "eto_ofr_desc": "", "eto_ofr_mcat_id": 655167,
                "glcat_mcat_name": "Penis Enlargement Product",
                "specifications": "Form:Capsule | Packaging Type:Bottle | Quantity:1 | Quantity Unit:Pack",
                "attribute_source": "230 | 230 | 1 | 2",
            },
        ]

    for bl in _test_data:
        print(f"\n{'='*65}")
        print(f"ID    : {bl['display_id']}")
        print(f"Title : {bl['eto_ofr_title']}")
        print(f"MCAT  : {bl.get('glcat_mcat_name','null')} (id={bl.get('eto_ofr_mcat_id',0)})")
        print(f"Query : {build_query(bl)}")

        fs       = get_field_sources(bl.get("specifications",""), bl.get("attribute_source",""))
        buyer_fs = {k: v["value"] for k,v in fs.items() if v["source_type"] in ("BUYER","AGENT")}
        print(f"Buyer specs: {buyer_fs}")

        suggestions, flag = run_mcat_ranker(
            bl, bi_encoder, cross_encoder, faiss_index, mcat_metadata
        )

        print(f"\nTop-{TOP_K_RETURN} suggestions:")
        for i, s in enumerate(suggestions, 1):
            marker = " ← SUGGESTED" if i == 1 and flag else ""
            print(f"  {i}. [{s['final_score']:.3f}] {s['full_path']}{marker}")

        if flag:
            print(f"\n🔴 MCAT_MISMATCH [{flag.severity}] confidence={flag.confidence:.0%}")
            print(f"   {flag.title}")
            print(f"   Evidence: {flag.evidence[:200]}")
            print(f"   Action  : {flag.suggested_action[:150]}")
        else:
            print(f"\n✅ MCAT mapping looks correct")

🔄 Loading ranker models...
🔄 Loading bi-encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Loading cross-encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Loading FAISS index...
✅ Ranker ready — 98,239 MCATs indexed
✅ LLM Gateway ready: anthropic/claude-sonnet-4-6

✅ Testing on first 5 real BuyLeads...


ID    : 144097681163
Title : Cotton Carry Bags
MCAT  : Cloth Bags (id=25024)
Query : Cotton Carry Bags Printed
Buyer specs: {'Bag Type': 'Printed', 'Buyer Filled Details': 'Buyer needs customized design with print.', 'Material': 'Cotton', 'Quantity': '100', 'Quantity Unit': 'Piece'}

Top-3 suggestions:
  1. [0.884] Bags, Handbags, Luggage Bags, Belts, Wallets and Accessories > Cotton Bags, Canvas Bags, Jute Bags & Other Fabric Bags > Cotton Bags > Printed Cotton Bag ← SUGGESTED
  2. [0.854] Bags, Handbags, Luggage Bags, Belts, Wallets and Accessories > Cotton Bags, Canvas Bags, Jute Bags & Other Fabric Bags > Cotton Bags > Cotton Carry Bag
  3. [0.851] Bags, Handbags, Luggage Bags, Belts, Wallets and Accessories > Cotton Bags, Canvas Bags, Jute Bags & Other Fabric Bags > Fabric Bag > Printed Cloth Bag

🔴 MCAT_MISMATCH [CRITICAL] confid

In [67]:
# %%writefile /content/drive/MyDrive/buylead_rca/07_scorer.py

import re
import math

# ══════════════════════════════════════════════════════════════
# SCORER — Computes quality score from Rule Engine + MCAT Ranker flags
#
# CALLED FROM: 08_rca_agent.py Stage 3
# RUNS AFTER:  Stage 1 (Rule Engine) + Stage 2 (MCAT Ranker)
# RUNS BEFORE: Stage 4 (LLM Reasoner)
#
# HOW IT WORKS:
#   Each check gets its own score out of its max marks.
#   Final score = sum of all earned marks / total max × 100
#
# SCORE TABLE:
# ┌──────────────────────────────────┬─────┬──────────────────────┐
# │ Check                            │ Max │ Source               │
# ├──────────────────────────────────┼─────┼──────────────────────┤
# │ MCAT correctly mapped            │ 30  │ 03_mcat_ranker       │
# │ ISQ fields filled by buyer       │ 15  │ 02_rule_engine       │
# │ ISQ completeness (filled/total)  │ 15  │ 02_rule_engine       │
# │ Title quality                    │ 15  │ 02_rule_engine       │
# │ Attribute source quality         │ 10  │ 02_rule_engine       │
# │ Description quality              │  5  │ 02_rule_engine       │
# │ Quantity sanity                  │  5  │ 02_rule_engine       │
# │ No prohibited/spam content       │  5  │ 02_rule_engine       │
# ├──────────────────────────────────┼─────┼──────────────────────┤
# │ TOTAL                            │ 100 │                      │
# └──────────────────────────────────┴─────┴──────────────────────┘
#
# QUALITY BANDS:
#   80-100 → EXCELLENT  Route to premium suppliers immediately
#   60-79  → GOOD       Route normally
#   40-59  → MEDIUM     Send reminder to buyer, route with lower priority
#   20-39  → POOR       Hold routing, trigger remediation
#   0-19   → CRITICAL   Block for human review
# ══════════════════════════════════════════════════════════════

QUALITY_BANDS = [
    (80, 100, "EXCELLENT", "Route to premium suppliers immediately."),
    (60,  79, "GOOD",      "Route normally."),
    (40,  59, "MEDIUM",    "Send reminder to buyer; route with lower priority."),
    (20,  39, "POOR",      "Hold routing; trigger remediation flow."),
    ( 0,  19, "CRITICAL",  "Block for human review."),
]

SUPPLIER_MATCH_IMPACT = {
    "EXCELLENT": "HIGH — all signals consistent, BuyLead will match right suppliers.",
    "GOOD":      "GOOD — minor gaps exist but matching quality is acceptable.",
    "MEDIUM":    "REDUCED — quality issues will limit supplier relevance.",
    "POOR":      "LOW — wrong routing likely due to data quality issues.",
    "CRITICAL":  "ZERO — severely miscategorized or spam; no useful matching.",
}


# ══════════════════════════════════════════════════════════════
# INDIVIDUAL CHECK SCORERS
# Each returns: (earned, max, reason_string)
# ══════════════════════════════════════════════════════════════

def score_mcat_mapping(flags, mcat_ranker_result=None):
    """
    Max: 30 pts
    Source: MCAT_MISMATCH or MCAT_UNMAPPED from 03_mcat_ranker / 02_rule_engine

    30  → MCAT is correct (no flag)
    0   → MCAT completely wrong (high confidence mismatch)
    15  → MCAT possibly wrong (low confidence mismatch)
    0   → MCAT not mapped at all (id=0)

    Why 30 pts (highest weight):
      Wrong MCAT = routed to wrong suppliers = zero conversions.
      Most critical quality dimension.
    """
    MAX = 30

    unmapped = next((f for f in flags if f.flag_id == "MCAT_UNMAPPED"), None)
    if unmapped:
        return 0, MAX, "MCAT not mapped (id=0) — BuyLead cannot be routed to any supplier"

    mismatch = next((f for f in flags if f.flag_id == "MCAT_MISMATCH"), None)
    if mismatch:
        # Higher confidence mismatch = lower score
        earned = round((1 - mismatch.confidence) * MAX, 1)
        return earned, MAX, (
            f"MCAT mismatch detected — confidence {mismatch.confidence:.0%} wrong. "
            f"Suggested: {mismatch.suggested_action[:60]}"
        )

    # No mismatch flag — check ranker top score if available
    if mcat_ranker_result and "top_score" in mcat_ranker_result:
        top = mcat_ranker_result["top_score"]
        if top < 0.25:
            return MAX, MAX, f"MCAT mapping not verified (brand name title, score={top:.3f}) — trusting existing"
        return MAX, MAX, f"MCAT mapping looks correct (ranker score={top:.3f})"

    return MAX, MAX, "MCAT mapping looks correct"


def score_isq_buyer_filled(flags, bl):
    """
    Max: 15 pts
    Source: NO_BUYER_FILLED_ISQS, MOSTLY_PREDICTED_ISQS from 02_rule_engine

    15  → Buyer actively filled ISQ fields
    0   → Buyer filled nothing despite being shown questions
    7   → Buyer filled some but mostly predicted

    Why matters:
      Buyer-filled values are most reliable.
      Predicted values may be wrong → bad supplier matching.
    """
    MAX       = 15
    isq_shown = bl.get("available_isq_at_approval") or 0

    if isq_shown == 0:
        return MAX, MAX, "No ISQs were shown to buyer — not applicable"

    no_buyer = next((f for f in flags if f.flag_id == "NO_BUYER_FILLED_ISQS"), None)
    if no_buyer:
        return 0, MAX, (
            f"Buyer was shown {isq_shown} ISQ questions but filled NONE. "
            f"All fields are auto/predicted — low data reliability."
        )

    mostly_pred = next((f for f in flags if f.flag_id == "MOSTLY_PREDICTED_ISQS"), None)
    if mostly_pred:
        return round(MAX * 0.5, 1), MAX, (
            f"Buyer shown {isq_shown} ISQs but filled very few. "
            f"Most values are model predictions."
        )

    return MAX, MAX, f"Buyer actively filled ISQ fields ({isq_shown} shown)"


def score_isq_completeness(flags, bl):
    """
    Max: 15 pts
    Source: MISSING_ISQ_FIELDS from 02_rule_engine

    15  → All required fields for this MCAT are filled
    0   → Less than 50% required fields filled
    Pro-rated for partial completeness.

    Why matters:
      Missing ISQ fields mean suppliers can't understand exact requirement.
    """
    MAX       = 15
    isq_shown = bl.get("available_isq_at_approval") or 0

    if isq_shown == 0:
        return MAX, MAX, "No ISQs shown to buyer — not applicable"

    flag = next((f for f in flags if f.flag_id == "MISSING_ISQ_FIELDS"), None)
    if not flag:
        return MAX, MAX, "All required ISQ fields are filled"

    match = re.search(r"Completeness: (\d+)/(\d+)", flag.evidence)
    if match:
        filled, total = int(match.group(1)), int(match.group(2))
        ratio  = filled / total
        earned = round(ratio * MAX, 1)
        return earned, MAX, (
            f"{filled}/{total} required ISQ fields filled. "
            f"Missing fields reduce supplier matching quality."
        )

    return round(MAX * 0.5, 1), MAX, "Some required ISQ fields missing"


def score_title_quality(flags):
    """
    Max: 15 pts
    Source: EMPTY_TITLE, SPAM_TITLE, GENERIC_TITLE, VAGUE_TITLE,
            B2C_TITLE_LANGUAGE, KEYWORD_STUFFED_TITLE, TITLE_TOO_LONG

    15  → Title is clear, specific, B2B appropriate
    0   → Title is empty, spam, or completely generic

    Why matters:
      Title is the primary search signal for matching suppliers.
    """
    MAX = 15

    critical = next((f for f in flags if f.flag_id in
                     ("EMPTY_TITLE", "SPAM_TITLE", "GENERIC_TITLE")), None)
    if critical:
        return 0, MAX, f"Critical title issue: {critical.flag_id} — {critical.title}"

    bad_flags = [f for f in flags if f.flag_id in (
        "VAGUE_TITLE", "B2C_TITLE_LANGUAGE",
        "KEYWORD_STUFFED_TITLE", "TITLE_TOO_LONG",
    )]
    if not bad_flags:
        return MAX, MAX, "Title is clear, specific, and B2B appropriate"

    total_penalty = sum(abs(f.score_impact) for f in bad_flags)
    earned        = max(0, round(MAX - (total_penalty * MAX / 15), 1))
    issues        = ", ".join(f.flag_id for f in bad_flags)
    return earned, MAX, f"Minor title issues: {issues}"


def score_attribute_source_quality(flags):
    """
    Max: 10 pts
    Source: SUSPICIOUS_AUTO_ISQ_VALUES, MEANINGLESS_AUTO_ISQ_VALUES,
            BUYER_B2C_VS_PREDICTED_B2B from 02_rule_engine

    10  → All auto/predicted values are consistent with title
    0   → Auto values contradict title OR buyer says B2C but predicted B2B

    Why matters:
      Wrong auto-filled values mislead supplier matching silently.
    """
    MAX = 10

    suspicious  = next((f for f in flags if f.flag_id == "SUSPICIOUS_AUTO_ISQ_VALUES"),  None)
    meaningless = next((f for f in flags if f.flag_id == "MEANINGLESS_AUTO_ISQ_VALUES"), None)
    b2c_b2b     = next((f for f in flags if f.flag_id == "BUYER_B2C_VS_PREDICTED_B2B"),  None)

    if suspicious and b2c_b2b:
        return 0, MAX, "Auto values contradict title AND buyer intent contradicts prediction"
    if suspicious:
        return round(MAX * 0.3, 1), MAX, f"Auto/predicted ISQ values inconsistent with title"
    if b2c_b2b:
        return round(MAX * 0.2, 1), MAX, "Buyer filled personal-use details but predicted Business Use"
    if meaningless:
        return round(MAX * 0.6, 1), MAX, "Some auto-filled values are meaningless (N/A, Other, etc.)"

    return MAX, MAX, "Auto/predicted ISQ values are consistent with title"


def score_description_quality(flags):
    """
    Max: 5 pts
    Source: DESCRIPTION_COPY_OF_TITLE, VAGUE_DESCRIPTION,
            B2C_DESCRIPTION, CONTACT_IN_DESCRIPTION

    5   → Description empty (fine — LLM generates it) OR good quality
    0   → Description exists but is B2C, contains contact info, or boilerplate

    Note: Empty description = full marks.
          LLM auto-generates description in Stage 4.
    """
    MAX = 5

    bad_ids = {
        "DESCRIPTION_COPY_OF_TITLE", "VAGUE_DESCRIPTION",
        "B2C_DESCRIPTION", "CONTACT_IN_DESCRIPTION",
        "LLM_CONTRADICTION_TITLE_DESC",
    }
    bad_flags = [f for f in flags if f.flag_id in bad_ids]
    if not bad_flags:
        return MAX, MAX, "Description OK (empty = fine, LLM will generate)"

    earned = max(0, round(MAX - len(bad_flags) * 2, 1))
    issues = ", ".join(f.flag_id for f in bad_flags)
    return earned, MAX, f"Description issues: {issues}"


def score_quantity_sanity(flags):
    """
    Max: 5 pts
    Source: ZERO_QUANTITY, UNREALISTIC_QUANTITY,
            HIGH_QTY_FOR_HEAVY_PRODUCT, UNREALISTIC_BULK_QUANTITY

    5   → Quantity is realistic for the product
    0   → Zero, negative, or absurdly high quantity
    """
    MAX = 5

    qty_flags = [f for f in flags if f.flag_id in (
        "ZERO_QUANTITY", "UNREALISTIC_QUANTITY",
        "HIGH_QTY_FOR_HEAVY_PRODUCT", "UNREALISTIC_BULK_QUANTITY",
    )]
    if not qty_flags:
        return MAX, MAX, "Quantity is realistic"

    return 0, MAX, qty_flags[0].title


def score_content_safety(flags):
    """
    Max: 5 pts
    Source: PROHIBITED_CONTENT, CONTACT_IN_TITLE, SPAM_TITLE

    5   → No prohibited content or contact info
    0   → Contains prohibited words or phone/email in title
    """
    MAX = 5

    bad_flags = [f for f in flags if f.flag_id in (
        "PROHIBITED_CONTENT", "CONTACT_IN_TITLE", "CONTACT_IN_DESCRIPTION",
    )]
    if not bad_flags:
        return MAX, MAX, "No prohibited content or contact info"

    return 0, MAX, f"Content safety issues: {[f.flag_id for f in bad_flags]}"


# ══════════════════════════════════════════════════════════════
# MAIN SCORER
# ══════════════════════════════════════════════════════════════

def get_quality_band(score):
    for low, high, band, advice in QUALITY_BANDS:
        if low <= score <= high:
            return band, advice
    return "CRITICAL", QUALITY_BANDS[-1][3]


def run_scorer(bl, all_flags, mcat_ranker_result=None):
    """
    Computes final BuyLead quality score from all flags.

    Called after Stage 1 (Rule Engine) + Stage 2 (MCAT Ranker).
    Score is fully deterministic — does NOT use LLM.

    Each check earns points out of its maximum.
    Final score = (total earned / 100) × 100

    Returns full scoring report with:
      - quality_score: 0-100
      - quality_band: EXCELLENT/GOOD/MEDIUM/POOR/CRITICAL
      - score_breakdown: per-check score with reason explaining WHY
      - rca_flags: sorted by severity
    """

    # Compute each check independently
    mcat_earned,  mcat_max,  mcat_reason  = score_mcat_mapping(all_flags, mcat_ranker_result)
    buyer_earned, buyer_max, buyer_reason = score_isq_buyer_filled(all_flags, bl)
    isq_earned,   isq_max,   isq_reason   = score_isq_completeness(all_flags, bl)
    title_earned, title_max, title_reason = score_title_quality(all_flags)
    attr_earned,  attr_max,  attr_reason  = score_attribute_source_quality(all_flags)
    desc_earned,  desc_max,  desc_reason  = score_description_quality(all_flags)
    qty_earned,   qty_max,   qty_reason   = score_quantity_sanity(all_flags)
    safe_earned,  safe_max,  safe_reason  = score_content_safety(all_flags)

    # Build breakdown
    breakdown = {
        "mcat_mapping": {
            "earned": mcat_earned,  "max": mcat_max,
            "score":  round(mcat_earned / mcat_max * 100, 1),
            "note":   mcat_reason,
        },
        "isq_buyer_filled": {
            "earned": buyer_earned, "max": buyer_max,
            "score":  round(buyer_earned / buyer_max * 100, 1) if buyer_max else 100,
            "note":   buyer_reason,
        },
        "isq_completeness": {
            "earned": isq_earned,   "max": isq_max,
            "score":  round(isq_earned / isq_max * 100, 1) if isq_max else 100,
            "note":   isq_reason,
        },
        "title_quality": {
            "earned": title_earned, "max": title_max,
            "score":  round(title_earned / title_max * 100, 1),
            "note":   title_reason,
        },
        "attribute_source": {
            "earned": attr_earned,  "max": attr_max,
            "score":  round(attr_earned / attr_max * 100, 1),
            "note":   attr_reason,
        },
        "description_quality": {
            "earned": desc_earned,  "max": desc_max,
            "score":  round(desc_earned / desc_max * 100, 1),
            "note":   desc_reason,
        },
        "quantity_sanity": {
            "earned": qty_earned,   "max": qty_max,
            "score":  round(qty_earned / qty_max * 100, 1),
            "note":   qty_reason,
        },
        "content_safety": {
            "earned": safe_earned,  "max": safe_max,
            "score":  round(safe_earned / safe_max * 100, 1),
            "note":   safe_reason,
        },
    }

    # Final score = total earned out of 100
    total_earned = (
        mcat_earned + buyer_earned + isq_earned + title_earned +
        attr_earned + desc_earned  + qty_earned + safe_earned
    )
    total_max = (
        mcat_max + buyer_max + isq_max + title_max +
        attr_max + desc_max  + qty_max + safe_max
    )
    final_score  = round((total_earned / total_max) * 100, 1)
    quality_band, band_advice = get_quality_band(final_score)

    # Sort flags by severity
    severity_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
    sorted_flags   = sorted(all_flags, key=lambda f: severity_order.get(f.severity, 9))

    return {
        "quality_score":   final_score,
        "quality_band":    quality_band,
        "band_advice":     band_advice,
        "total_earned":    total_earned,
        "total_max":       total_max,
        "score_formula":   f"{total_earned}/{total_max} = {final_score}/100",
        "score_breakdown": breakdown,
        "rca_flags": [
            {
                "flag_id":          f.flag_id,
                "severity":         f.severity,
                "confidence":       f.confidence,
                "title":            f.title,
                "evidence":         f.evidence,
                "suggested_action": f.suggested_action,
                "score_impact":     f.score_impact,
                "source":           f.source,
            }
            for f in sorted_flags
        ],
        "supplier_match_impact": SUPPLIER_MATCH_IMPACT[quality_band],
        "remediation_priority": (
            "IMMEDIATE" if quality_band in ("CRITICAL", "POOR") else
            "HIGH"      if quality_band == "MEDIUM" else
            "NORMAL"
        ),
    }


# ══════════════════════════════════════════════════════════════
# TEST
# ══════════════════════════════════════════════════════════════
print("Testing scorer with mock flags...\n")

mock_bl = {
    "eto_ofr_title":   "Plastic 4dx JCB Excavator Toy",
    "glcat_mcat_name": "JCB Excavator",
    "available_isq_at_approval": 5,
}

mock_flags = [
    RCAFlag("MCAT_MISMATCH",      "CRITICAL", 0.87, "Wrong MCAT",
            "...", "Remap to Construction Toy", -35, "mcat_ranker"),
    RCAFlag("MISSING_ISQ_FIELDS", "HIGH",     1.00, "3 ISQ fields missing",
            "Completeness: 2/5. Missing: ['Brand','Condition','Operating Weight'].",
            "Prompt buyer.", -12, "rule_engine"),
    RCAFlag("NO_BUYER_FILLED_ISQS","HIGH",    0.95, "No buyer ISQ",
            "5 ISQ questions shown. All 3 filled by system.",
            "Re-engage buyer.", -15, "rule_engine"),
]

result = run_scorer(mock_bl, mock_flags, mcat_ranker_result={"top_score": 0.87})

print(f"Formula : {result['score_formula']}")
print(f"Score   : {result['quality_score']}/100  [{result['quality_band']}]")
print(f"Advice  : {result['band_advice']}")
print(f"\nScore Breakdown (earned / max = % and why):")
print(f"  {'Check':<22} {'Earned':>6} {'Max':>4} {'Score%':>7}  Reason")
print(f"  {'-'*80}")
for k, v in result["score_breakdown"].items():
    status = "✅" if v["score"] >= 80 else ("⚠️ " if v["score"] >= 40 else "❌")
    print(f"  {status} {k:<22} {v['earned']:>6.1f} {v['max']:>4}  {v['score']:>6.1f}%  {v['note'][:45]}")
print(f"  {'-'*80}")
print(f"  {'TOTAL':<22} {result['total_earned']:>6.1f} {result['total_max']:>4}  {result['quality_score']:>6.1f}%")
print(f"\nSupplier impact: {result['supplier_match_impact']}")

Testing scorer with mock flags...

Formula : 49.9/100 = 49.9/100
Score   : 49.9/100  [MEDIUM]
Advice  : Send reminder to buyer; route with lower priority.

Score Breakdown (earned / max = % and why):
  Check                  Earned  Max  Score%  Reason
  --------------------------------------------------------------------------------
  ❌ mcat_mapping              3.9   30    13.0%  MCAT mismatch detected — confidence 87% wrong
  ❌ isq_buyer_filled          0.0   15     0.0%  Buyer was shown 5 ISQ questions but filled NO
  ⚠️  isq_completeness          6.0   15    40.0%  2/5 required ISQ fields filled. Missing field
  ✅ title_quality            15.0   15   100.0%  Title is clear, specific, and B2B appropriate
  ✅ attribute_source         10.0   10   100.0%  Auto/predicted ISQ values are consistent with
  ✅ description_quality       5.0    5   100.0%  Description OK (empty = fine, LLM will genera
  ✅ quantity_sanity           5.0    5   100.0%  Quantity is realistic
  ✅ content_safety   

In [68]:
# %%writefile /content/drive/MyDrive/buylead_rca/08_rca_agent.py

import json
import time
import traceback

# ── NOTE ─────────────────────────────────────────────────────
# Run cells in order before this:
#   01_data_prep    → parse_specifications, get_mandatory_isqs
#   02_rule_engine  → RCAFlag, run_rule_engine
#   03_mcat_ranker  → run_mcat_ranker, bi_encoder, faiss_index, mcat_metadata
#   06_llm_reasoner → gemini_model, detect_contradictions_llm,
#                     generate_description, generate_rca_narrative
#   07_scorer       → run_scorer
# ─────────────────────────────────────────────────────────────

# ══════════════════════════════════════════════════════════════
# PIPELINE ORDER (correct):
#
#  Stage 1 → Rule Engine
#    Runs fast deterministic checks:
#    - Title quality, spam, prohibited content
#    - ISQ completeness (DB query for required fields)
#    - Attribute source quality (buyer vs predicted fill rate)
#    - Quantity sanity
#    - Contact info, MCAT unmapped
#    Output: list of RCAFlags with score_impact
#
#  Stage 2 → MCAT Ranker
#    Checks if mapped MCAT is correct:
#    - Signal 1: title embedding vs 98k MCAT index
#    - Signal 2: buyer-filled specs vs MCAT
#    - LLM fallback for brand names (auto, via gateway)
#    Output: top-3 MCAT suggestions + optional MCAT_MISMATCH flag
#
#  Stage 3 → Scorer
#    Combines all flags from Stage 1 + Stage 2:
#    - Each check gets its own score (0-100)
#    - Final score = weighted combination
#    - Assigns quality band: EXCELLENT/GOOD/MEDIUM/POOR/CRITICAL
#    - Explains WHY each component scored what it did
#    Output: quality_score, score_breakdown, quality_band
#
#  Stage 4 → LLM Reasoner (ONLY after scoring)
#    Uses LLM Gateway only for:
#    - Contradiction detection (catches subtle issues rules miss)
#    - Auto-generate description if empty
#    - Write human-readable narrative for ops team
#    NOT used for scoring — LLM runs AFTER score is computed
#    Output: narrative, auto_description, additional contradiction flags
#
# WHY THIS ORDER:
#   Score is based on rule engine + MCAT ranker outputs — deterministic,
#   fast, explainable. LLM adds richness (narrative, description, deep
#   contradictions) but does not change the core score.
#   This keeps scoring transparent and auditable.
# ══════════════════════════════════════════════════════════════


def analyze_buylead(bl, use_llm=True, verbose=True):
    """
    Full RCA analysis for one BuyLead.

    Pipeline:
      Stage 1: Rule Engine      → flags from deterministic checks
      Stage 2: MCAT Ranker      → MCAT correctness check
      Stage 3: Scorer           → quality score + breakdown with reasons
      Stage 4: LLM Reasoner     → narrative + description (after scoring)

    Returns complete RCA result dict with score, flags, breakdown, narrative.
    """
    start     = time.time()
    bl_id     = bl.get("display_id", "unknown")
    all_flags = []

    if verbose:
        print(f"\n{'='*60}")
        print(f"📋 BuyLead {bl_id}: '{bl.get('eto_ofr_title', '')}'")
        print(f"   MCAT: {bl.get('glcat_mcat_name','')} (id={bl.get('eto_ofr_mcat_id',0)})")

    # ══════════════════════════════════════════════════════════
    # STAGE 1 — RULE ENGINE
    # Fast deterministic checks. Each flag has:
    #   - flag_id: what check failed
    #   - severity: CRITICAL/HIGH/MEDIUM/LOW
    #   - evidence: exactly why it fired
    #   - score_impact: how many points this costs
    # ══════════════════════════════════════════════════════════
    try:
        rule_flags = run_rule_engine(bl)
        all_flags.extend(rule_flags)
        if verbose:
            print(f"\n  [Stage 1] Rule Engine: {len(rule_flags)} flag(s)")
            for f in rule_flags:
                icon = {"CRITICAL":"🔴","HIGH":"🟠","MEDIUM":"🟡","LOW":"🔵"}.get(f.severity,"⚪")
                print(f"    {icon} {f.flag_id}: {f.title}")
                print(f"       Reason: {f.evidence[:100]}")
                print(f"       Score impact: {f.score_impact} pts")
    except Exception as e:
        if verbose: print(f"  ✗ Rule engine error: {e}")

    # Early exit for spam/prohibited — no point running ML
    blocking = [f for f in all_flags if f.flag_id in
                ("SPAM_TITLE", "PROHIBITED_CONTENT", "LOW_ENTROPY_TITLE")]

    mcat_suggestions   = []
    mcat_ranker_result = None
    auto_description   = None
    narrative_result   = None

    if blocking:
        if verbose:
            print(f"\n  ⚠️  Spam/prohibited detected — skipping ML stages")
        narrative_result = {
            "narrative": "BuyLead flagged as spam or prohibited content.",
            "priority":  "IMMEDIATE"
        }
        # Still compute score so result dict is always complete
        score_result = run_scorer(bl, all_flags, mcat_ranker_result=None)
        if verbose:
            print(f"\n  [Stage 3] Scorer (spam path):")
            print(f"    Score: {score_result['quality_score']}/100 [{score_result['quality_band']}]")

    else:
        # ══════════════════════════════════════════════════════
        # STAGE 2 — MCAT RANKER
        # Checks if the mapped MCAT is correct for this title+specs.
        # Uses embedding search (fast) + LLM fallback (for brand names).
        # Adds MCAT_MISMATCH flag if wrong, with suggested correct MCAT.
        # ══════════════════════════════════════════════════════
        try:
            mcat_suggestions, mcat_flag = run_mcat_ranker(
                bl, bi_encoder, cross_encoder, faiss_index, mcat_metadata
            )
            if mcat_flag:
                all_flags.append(mcat_flag)
            if mcat_suggestions:
                mcat_ranker_result = {"top_score": mcat_suggestions[0]["final_score"]}

            if verbose:
                print(f"\n  [Stage 2] MCAT Ranker:")
                print(f"    Mapped:  {bl.get('glcat_mcat_name','')} (id={bl.get('eto_ofr_mcat_id',0)})")
                if mcat_suggestions:
                    print(f"    Top-3 suggestions:")
                    for i, s in enumerate(mcat_suggestions, 1):
                        print(f"      {i}. [{s['final_score']:.3f}] {s['mcat_name']}")
                if mcat_flag:
                    print(f"    🔴 MCAT_MISMATCH: {mcat_flag.title}")
                    print(f"       Confidence: {mcat_flag.confidence:.0%}")
                    print(f"       Score impact: {mcat_flag.score_impact} pts")
                else:
                    print(f"    ✅ MCAT mapping looks correct")
        except Exception as e:
            if verbose: print(f"  ✗ MCAT ranker error: {e}")
            traceback.print_exc()

        # ══════════════════════════════════════════════════════
        # STAGE 3 — SCORER
        # Combines ALL flags from Stage 1 + Stage 2.
        # Computes score for each component with explanation:
        #
        #   Title quality:      X/100 — "Issues: VAGUE_TITLE, B2C_LANGUAGE"
        #   ISQ completeness:   X/100 — "2/5 required fields filled"
        #   MCAT alignment:     X/100 — "Mismatch detected (confidence 94%)"
        #   Description:        X/100 — "Description OK"
        #   Quantity sanity:    X/100 — "Quantity looks realistic"
        #
        # Final score = weighted combination of all components.
        # ══════════════════════════════════════════════════════
        score_result = run_scorer(bl, all_flags, mcat_ranker_result)

        if verbose:
            print(f"\n  [Stage 3] Scorer:")
            print(f"    Final Score: {score_result['quality_score']}/100 [{score_result['quality_band']}]")
            print(f"    {score_result['band_advice']}")
            print(f"\n    Score Breakdown (why each component scored this):")
            for k, v in score_result["score_breakdown"].items():
                bar     = "█" * int(v["score"]/10) + "░" * (10 - int(v["score"]/10))
                status  = "✅" if v["score"] >= 70 else ("⚠️ " if v["score"] >= 40 else "❌")
                print(f"    {status} {k:<24} {v['score']:>5.1f}/100  {bar}  {v['note'][:45]}")

        # ══════════════════════════════════════════════════════
        # STAGE 4 — LLM REASONER (runs AFTER scoring)
        # Only runs if:
        #   a) use_llm=True AND gemini_model is available
        #   b) There are flags to analyze (don't waste API calls on clean BLs)
        #
        # Does 3 things:
        #   1. Contradiction detection — catches deep semantic issues
        #      that rule engine and scorer missed
        #   2. Auto-generate description if empty
        #   3. Write human-readable narrative explaining the score
        #
        # NOTE: LLM does NOT change the score.
        #       Score is already computed in Stage 3.
        #       LLM only adds explanation and enrichment.
        # ══════════════════════════════════════════════════════
        if use_llm:
            try:
                _llm = gemini_model
            except NameError:
                _llm = None

            if _llm is not None:
                if verbose:
                    print(f"\n  [Stage 4] LLM Reasoner (Gateway):")

                # 4a. Contradiction detection (only if there are already flags)
                if all_flags:
                    try:
                        llm_flags    = detect_contradictions_llm(bl, _llm)
                        existing_ids = {f.flag_id for f in all_flags}
                        new_flags    = [f for f in llm_flags
                                        if f.flag_id not in existing_ids]
                        all_flags.extend(new_flags)
                        if verbose:
                            print(f"    Contradiction check: {len(new_flags)} new flag(s)")
                            for f in new_flags:
                                print(f"      🤖 {f.flag_id}: {f.title[:60]}")
                    except Exception as e:
                        if verbose: print(f"    ✗ Contradiction detection: {e}")

                # 4b. Auto-generate description if empty
                try:
                    auto_description = generate_description(bl, _llm)
                    if auto_description and verbose:
                        print(f"    ✅ Description auto-generated")
                except Exception as e:
                    if verbose: print(f"    ✗ Description gen: {e}")

                # 4c. Generate narrative AFTER score is known
                try:
                    narrative_result = generate_rca_narrative(
                        bl, all_flags,
                        score_result["quality_score"],
                        _llm
                    )
                    if verbose and narrative_result:
                        print(f"    ✅ Narrative generated")
                        print(f"    📋 {narrative_result['narrative'][:120]}...")
                        print(f"    Priority: {narrative_result['priority']}")
                except Exception as e:
                    if verbose: print(f"    ✗ Narrative gen: {e}")
            else:
                if verbose:
                    print(f"\n  [Stage 4] LLM Reasoner: skipped (gemini_model not set)")

    # ══════════════════════════════════════════════════════════
    # ASSEMBLE FINAL OUTPUT
    # ══════════════════════════════════════════════════════════
    elapsed_ms = round((time.time() - start) * 1000)

    # Sort flags by severity for output
    severity_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
    sorted_flags   = sorted(all_flags,
                            key=lambda f: severity_order.get(f.severity, 9))

    result = {
        "buylead_id":   bl_id,
        "title":        bl.get("eto_ofr_title", ""),
        "mapped_mcat":  {
            "id":   bl.get("eto_ofr_mcat_id"),
            "name": bl.get("glcat_mcat_name", ""),
        },
        "suggested_mcats": [
            {
                "rank":       i + 1,
                "mcat_id":    s["mcat_id"],
                "mcat_name":  s["mcat_name"],
                "full_path":  s["full_path"],
                "confidence": round(s["final_score"], 3),
            }
            for i, s in enumerate(mcat_suggestions[:3])
        ],
        "quality_score":              score_result["quality_score"],
        "quality_band":               score_result["quality_band"],
        "band_advice":                score_result["band_advice"],
        "score_breakdown":            score_result["score_breakdown"],
        "rca_flags":                  [
            {
                "flag_id":          f.flag_id,
                "severity":         f.severity,
                "confidence":       f.confidence,
                "title":            f.title,
                "evidence":         f.evidence,
                "suggested_action": f.suggested_action,
                "score_impact":     f.score_impact,
                "source":           f.source,
            }
            for f in sorted_flags
        ],
        "supplier_match_impact":      score_result["supplier_match_impact"],
        "remediation_priority":       score_result["remediation_priority"],
        "auto_generated_description": auto_description,
        "rca_narrative":              narrative_result.get("narrative") if narrative_result else None,
        "llm_priority":               narrative_result.get("priority")  if narrative_result else None,
        "processing_time_ms":         elapsed_ms,
    }

    if verbose:
        print(f"\n  ⏱️  Total: {elapsed_ms}ms")

    return result


def analyze_batch(buyLeads, save_to=None, use_llm=True):
    """Process a list of BuyLeads. Optionally save results to JSON."""
    results = []
    total   = len(buyLeads)
    print(f"\n🔄 Processing {total} BuyLeads...")

    for i, bl in enumerate(buyLeads, 1):
        print(f"\n[{i}/{total}]", end="")
        try:
            results.append(analyze_buylead(bl, use_llm=use_llm))
        except Exception as e:
            print(f"  ✗ Failed: {e}")
            traceback.print_exc()
            results.append({
                "buylead_id": bl.get("display_id"),
                "error":      str(e),
            })

    if save_to:
        with open(save_to, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f"\n✅ Saved → {save_to}")

    return results


# ── Test ──────────────────────────────────────────────────────
try:
    _test_data = buyLeads[:3]
    print(f"Testing on first 3 real BuyLeads...")
except NameError:
    _test_data = [
        {
            "display_id":      1,
            "eto_ofr_title":   "Plastic 4dx JCB Excavator Toy",
            "eto_ofr_desc":    "",
            "eto_ofr_mcat_id": 6032,
            "glcat_mcat_name": "JCB Excavator",
            "specifications":  "Material:Plastic | Color:Yellow | Quantity:50 | Quantity Unit:Piece",
            "attribute_source":"214 | 214 | 212 | 213",
            "available_isq_at_approval": 5,
        },
        {
            "display_id":      2,
            "eto_ofr_title":   "Cotton Carry Bags",
            "eto_ofr_desc":    "",
            "eto_ofr_mcat_id": 25024,
            "glcat_mcat_name": "Cloth Bags",
            "specifications":  "Bag Type:Printed | Material:Cotton | Quantity:100 | Quantity Unit:Piece",
            "attribute_source":"214 | 214 | 212 | 213",
            "available_isq_at_approval": 5,
        },
    ]

_use_llm = True
try:
    _ = gemini_model
except NameError:
    _use_llm = False
    print("ℹ️  gemini_model not set — LLM stage will be skipped")

for bl in _test_data:
    result = analyze_buylead(bl, use_llm=_use_llm, verbose=True)

# Save batch
results = analyze_batch(
    _test_data,
    save_to="/content/drive/MyDrive/buylead_rca/outputs/rca_results.json",
    use_llm=_use_llm,
)
print(f"\n✅ Done.")

Testing on first 3 real BuyLeads...

📋 BuyLead 144097681163: 'Cotton Carry Bags'
   MCAT: Cloth Bags (id=25024)

  [Stage 1] Rule Engine: 2 flag(s)
    🟠 MISSING_ISQ_FIELDS: 3 required ISQ field(s) not filled
       Reason: 5 ISQ questions were shown to buyer. MCAT 25024 requires: ['Bag Type', 'Material', 'Color', 'Size', 
       Score impact: -12.0 pts
    🟠 NO_BUYER_FILLED_ISQS: No ISQ fields filled by buyer despite 5 questions shown
       Reason: 5 ISQ questions were shown to buyer at approval time. All 3 spec fields were filled by system (auto/
       Score impact: -15 pts

  [Stage 2] MCAT Ranker:
    Mapped:  Cloth Bags (id=25024)
    Top-3 suggestions:
      1. [0.884] Printed Cotton Bag
      2. [0.854] Cotton Carry Bag
      3. [0.851] Printed Cloth Bag
    🔴 MCAT_MISMATCH: MCAT 'Cloth Bags' appears incorrect — likely 'Printed Cotton Bag' based on TITLE + BUYER_SPECS
       Confidence: 95%
       Score impact: -35 pts

  [Stage 3] Scorer:
    Final Score: 47.5/100 [MEDIUM]
  

In [69]:
# %%writefile /content/drive/MyDrive/buylead_rca/09_demo_gradio.py

import json
import gradio as gr

# ── NOTE ─────────────────────────────────────────────────────
# Run all previous cells (01-08) before this one.
# analyze_buylead, run_scorer etc. are in Colab namespace.
# ─────────────────────────────────────────────────────────────

SEVERITY_EMOJI = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡", "LOW": "🔵"}
BAND_COLOR = {
    "EXCELLENT": "#2d8a4e", "GOOD": "#0f6e56",
    "MEDIUM": "#854f0b", "POOR": "#993c1d", "CRITICAL": "#a32d2d",
}


# ══════════════════════════════════════════════════════════════
# HTML FORMATTERS
# ══════════════════════════════════════════════════════════════

def fmt_score_bar(earned, max_score, label, note):
    pct   = int((earned / max_score) * 100) if max_score else 0
    color = "#2d8a4e" if pct > 70 else ("#854f0b" if pct > 40 else "#a32d2d")
    return f"""
    <div style="margin-bottom:12px">
      <div style="display:flex;justify-content:space-between;font-size:13px;margin-bottom:3px">
        <span style="font-weight:500">{label}</span>
        <span style="color:{color};font-weight:600">{earned}/{max_score} pts</span>
      </div>
      <div style="background:#e8e8e8;border-radius:4px;height:10px">
        <div style="background:{color};width:{pct}%;height:10px;border-radius:4px;transition:width 0.3s"></div>
      </div>
      <div style="font-size:11px;color:#666;margin-top:3px">{note[:80]}</div>
    </div>"""


def fmt_flag(flag):
    icon = SEVERITY_EMOJI.get(flag["severity"], "⚪")
    bg   = {"CRITICAL":"#fff0f0","HIGH":"#fff5ef","MEDIUM":"#fffaed","LOW":"#eff6ff"}.get(flag["severity"],"#f5f5f5")
    border = {"CRITICAL":"#e53e3e","HIGH":"#dd6b20","MEDIUM":"#d69e2e","LOW":"#3182ce"}.get(flag["severity"],"#ccc")
    return f"""
    <div style="background:{bg};border-radius:8px;padding:12px;margin-bottom:8px;border-left:4px solid {border}">
      <div style="font-weight:600;font-size:13px">{icon} [{flag['severity']}] {flag['flag_id']}</div>
      <div style="font-size:13px;color:#333;margin-top:4px">{flag['title']}</div>
      <div style="font-size:12px;color:#555;margin-top:4px"><b>Evidence:</b> {flag['evidence'][:200]}</div>
      <div style="font-size:12px;color:#1a6e3c;margin-top:4px">→ {flag['suggested_action'][:150]}</div>
      <div style="font-size:11px;color:#888;margin-top:4px">
        Confidence: {flag['confidence']:.0%} &nbsp;|&nbsp; Score impact: {flag['score_impact']} pts &nbsp;|&nbsp; Source: {flag['source']}
      </div>
    </div>"""


def fmt_score_card(result):
    band  = result["quality_band"]
    color = BAND_COLOR.get(band, "#444")
    score = result["quality_score"]
    html  = f"""
    <div style="text-align:center;padding:24px 0 16px">
      <div style="font-size:64px;font-weight:700;color:{color};line-height:1">{score}</div>
      <div style="font-size:20px;color:{color};font-weight:600;margin-top:4px">{band}</div>
      <div style="font-size:13px;color:#666;margin-top:6px">{result['band_advice']}</div>
      <div style="font-size:12px;color:#555;margin-top:6px;padding:8px;background:#f5f5f5;border-radius:6px;display:inline-block">
        {result['supplier_match_impact']}
      </div>
    </div>
    <hr style="margin:16px 0;border:none;border-top:1px solid #e0e0e0"/>
    <div style="font-size:13px;font-weight:600;margin-bottom:12px">Score Breakdown — {result.get('score_formula', str(result.get('total_earned','?')) + '/' + str(result.get('total_max',100)) + ' = ' + str(result.get('quality_score','?')) + '/100')}</div>
    """
    for k, v in result["score_breakdown"].items():
        label = k.replace("_", " ").title()
        html += fmt_score_bar(v["earned"], v["max"], label, v["note"])
    html += f"<p style='font-size:11px;color:#999;margin-top:12px'>⏱️ {result.get('processing_time_ms',0)}ms &nbsp;|&nbsp; {len(result['rca_flags'])} flag(s) &nbsp;|&nbsp; Priority: {result['remediation_priority']}</p>"
    return html


def fmt_mcat(result):
    mapped = result["mapped_mcat"]
    html   = f"<div style='font-size:13px'>"
    html  += f"<p><b>Currently mapped:</b> {mapped['name']} <span style='color:#888'>(id={mapped['id']})</span></p>"

    if result["suggested_mcats"]:
        html += "<p style='font-weight:600;margin-top:12px'>Top suggestions from MCAT dump (98k MCATs):</p>"
        for s in result["suggested_mcats"]:
            bar_w   = int(s["confidence"] * 100)
            is_same = s["mcat_name"].lower() == mapped["name"].lower()
            border  = "#2d8a4e" if is_same else "#0f6e56"
            html   += f"""
            <div style="margin-bottom:10px;padding:12px;background:#f7f7f7;border-radius:8px;border-left:3px solid {border}">
              <div style="font-weight:600">#{s['rank']} {s['mcat_name']}
                <span style="color:#666;font-size:11px;font-weight:normal"> (id={s['mcat_id']})</span>
                {'<span style="color:#2d8a4e;font-size:11px"> ✅ matches mapped</span>' if is_same else ''}
              </div>
              <div style="font-size:11px;color:#777;margin:2px 0">{s['full_path']}</div>
              <div style="background:#ddd;border-radius:3px;height:6px;margin-top:6px">
                <div style="background:#0f6e56;width:{bar_w}%;height:6px;border-radius:3px"></div>
              </div>
              <div style="font-size:11px;color:#555;margin-top:3px">Embedding confidence: {s['confidence']:.1%}</div>
            </div>"""
    else:
        html += "<p style='color:#888'>MCAT ranker did not return suggestions (MCAT unmapped or ranker skipped).</p>"
    html += "</div>"
    return html


def fmt_narrative(result):
    html = ""
    if result.get("rca_narrative"):
        priority_color = {"IMMEDIATE":"#e53e3e","HIGH":"#dd6b20","NORMAL":"#2d8a4e"}.get(
            result.get("llm_priority","NORMAL"), "#666")
        html += f"""
        <div style="padding:14px;background:#eef6ff;border-radius:8px;margin-bottom:12px;border-left:4px solid #3182ce">
          <div style="font-weight:600;font-size:13px;margin-bottom:6px">📋 RCA Narrative (LLM Generated)</div>
          <div style="font-size:13px;color:#333;line-height:1.6">{result['rca_narrative']}</div>
          <div style="font-size:12px;margin-top:8px">
            Priority: <span style="color:{priority_color};font-weight:600">{result.get('llm_priority','N/A')}</span>
          </div>
        </div>"""
    if result.get("auto_generated_description"):
        html += f"""
        <div style="padding:14px;background:#f0fff4;border-radius:8px;border-left:4px solid #2d8a4e">
          <div style="font-weight:600;font-size:13px;margin-bottom:6px">✍️ Auto-Generated Description</div>
          <div style="font-size:13px;color:#333;line-height:1.6">{result['auto_generated_description']}</div>
          <div style="font-size:11px;color:#888;margin-top:6px">Generated because original description was empty</div>
        </div>"""
    if not html:
        html = "<p style='color:#666;font-size:13px'>LLM stage was skipped or not needed.</p>"
    return html


# ══════════════════════════════════════════════════════════════
# SINGLE BUYLEAD ANALYSIS
# ══════════════════════════════════════════════════════════════
def run_single(title, desc, mcat_id, mcat_name, specs, attr_src, isq_shown):
    import traceback

    if not title.strip():
        err = "<p style='color:red;padding:20px'>⚠️ Please enter a title.</p>"
        return err, "", "", ""

    try:
        bl = {
            "display_id":                "demo_001",
            "eto_ofr_title":             title.strip(),
            "eto_ofr_desc":              (desc or "").strip(),
            "eto_ofr_mcat_id":           int(mcat_id) if str(mcat_id).strip().isdigit() else 0,
            "glcat_mcat_name":           (mcat_name or "").strip(),
            "specifications":            (specs or "").strip(),
            "attribute_source":          (attr_src or "").strip(),
            "available_isq_at_approval": int(isq_shown) if str(isq_shown).strip().isdigit() else 0,
            "image_url":                 None,
        }

        try:
            _llm = gemini_model
        except NameError:
            _llm = None

        result = analyze_buylead(bl, use_llm=(_llm is not None), verbose=False)

        return (
            fmt_score_card(result),
            "".join(fmt_flag(f) for f in result["rca_flags"]) or "<p style='color:#2d8a4e'>✅ No issues found!</p>",
            fmt_mcat(result),
            fmt_narrative(result),
        )

    except Exception as e:
        tb = traceback.format_exc()
        err_html = f"""
        <div style="padding:16px;background:#fff0f0;border-radius:8px;border-left:4px solid #e53e3e">
          <div style="font-weight:600;color:#e53e3e;font-size:14px">❌ Analysis Error</div>
          <div style="font-size:13px;color:#333;margin-top:8px"><b>Error:</b> {str(e)}</div>
          <details style="margin-top:8px">
            <summary style="font-size:12px;color:#666;cursor:pointer">Show full traceback</summary>
            <pre style="font-size:11px;color:#555;margin-top:6px;white-space:pre-wrap;background:#f9f9f9;padding:8px;border-radius:4px">{tb}</pre>
          </details>
          <div style="font-size:12px;color:#666;margin-top:8px">
            ℹ️ Make sure cells 01–07 ran successfully before running the demo.
          </div>
        </div>"""
        return err_html, err_html, err_html, err_html


# ══════════════════════════════════════════════════════════════
# BATCH JSON FILE ANALYSIS
# ══════════════════════════════════════════════════════════════
def run_batch(file_obj):
    import traceback

    if file_obj is None:
        return "<p style='color:red'>Please upload a JSON file.</p>", None

    try:
        filepath = file_obj.name
        # Use load_buyLeads from 01_data_prep for proper field normalization
        # This handles: null glcat_mcat_name, null attribute_source,
        # mcat_id=0, available_isq_at_approval=null etc.
        try:
            raw = load_buyLeads(filepath)
            if not raw:
                return "<p style='color:red'>No valid BuyLeads found after loading. Check required fields: display_id, eto_ofr_title, eto_ofr_mcat_id, specifications</p>", None
        except Exception as e1:
            # Fallback: try raw JSON parse
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    raw = json.load(f)
                if isinstance(raw, dict):
                    raw = [raw]
            except Exception as e2:
                return f"<p style='color:red'>❌ Failed to load file.<br>load_buyLeads error: {e1}<br>JSON parse error: {e2}</p>", None

    except Exception as e:
        tb = traceback.format_exc()
        return f"<p style='color:red'>❌ File error: {e}<br><pre>{tb[:500]}</pre></p>", None

    if not raw:
        return "<p style='color:red'>JSON file is empty or no valid records found.</p>", None

    try:
        _llm = gemini_model
    except NameError:
        _llm = None

    results   = []
    html_rows = ""
    band_counts = {"EXCELLENT":0,"GOOD":0,"MEDIUM":0,"POOR":0,"CRITICAL":0}

    # Limit to 50 BuyLeads in UI to avoid timeout
    # Full batch can be run via analyze_batch() in code
    MAX_UI_BATCH = 50
    display_raw  = raw[:MAX_UI_BATCH]
    if len(raw) > MAX_UI_BATCH:
        html_rows += f"<tr><td colspan='7' style='padding:8px;background:#fffaed;color:#854f0b;font-size:12px'>⚠️ File has {len(raw)} BuyLeads — showing first {MAX_UI_BATCH} in UI. Full results saved to JSON.</td></tr>"

    for i, bl in enumerate(display_raw):
        bl_id = bl.get("display_id", f"row_{i+1}")
        print(f"  [{i+1}/{len(display_raw)}] {bl_id}: {str(bl.get('eto_ofr_title',''))[:40]}...")
        try:
            # Run without LLM for speed in batch — LLM takes too long per record
            r = analyze_buylead(bl, use_llm=False, verbose=False)
            results.append(r)
            band_counts[r["quality_band"]] = band_counts.get(r["quality_band"], 0) + 1
            print(f"    ✅ {r['quality_score']}/100 [{r['quality_band']}] | {len(r['rca_flags'])} flags")

            band  = r["quality_band"]
            color = BAND_COLOR.get(band, "#444")
            flags_summary = ", ".join(
                f"{f['flag_id']}({f['severity'][0]})"
                for f in r["rca_flags"][:3]
            ) + ("..." if len(r["rca_flags"]) > 3 else "")

            # Build concise RCA summary from top flags
            # Note: r["rca_flags"] are dicts, not RCAFlag objects
            import re as _re
            rca_parts = []
            for _f in r["rca_flags"][:3]:
                _fid = _f["flag_id"]
                if _fid == "MCAT_MISMATCH":
                    _m = _re.search(r"Remap to MCAT \d+: '([^']+)'", _f["suggested_action"])
                    rca_parts.append(f"Remap → {_m.group(1)[:25] if _m else 'correct MCAT'}")
                elif _fid == "MCAT_UNMAPPED":
                    rca_parts.append("Assign MCAT category")
                elif _fid == "MISSING_ISQ_FIELDS":
                    _m2 = _re.search(r"Completeness: (\d+/\d+)", _f["evidence"])
                    rca_parts.append(f"Fill ISQ: {_m2.group(1) if _m2 else 'required fields'}")
                elif _fid == "NO_BUYER_FILLED_ISQS":
                    rca_parts.append("Re-engage buyer to fill ISQ")
                elif _fid == "EMPTY_TITLE":
                    rca_parts.append("Add product title")
                elif _fid == "TITLE_TOO_LONG":
                    rca_parts.append("Shorten title to 3-10 words")
                elif _fid == "VAGUE_TITLE":
                    rca_parts.append("Use specific product name")
                elif _fid == "SUSPICIOUS_AUTO_ISQ_VALUES":
                    rca_parts.append("Fix inconsistent auto-filled values")
                elif _fid == "BUYER_B2C_VS_PREDICTED_B2B":
                    rca_parts.append("Verify B2B intent")
                elif "CONTRADICTION" in _fid:
                    rca_parts.append(f"Fix: {_f['title'][:40]}")
                else:
                    rca_parts.append(_f["suggested_action"][:40])

            action_text = " | ".join(rca_parts) if rca_parts else "No action needed ✅"
            rca_color   = "#e53e3e" if band in ("CRITICAL","POOR") else ("#854f0b" if band == "MEDIUM" else "#2d8a4e")

            # Build RCA root cause — plain English, max 2 sentences
            _flag_ids = [f["flag_id"] for f in r["rca_flags"]]
            rca_sentences = []

            if "MCAT_UNMAPPED" in _flag_ids:
                rca_sentences.append("BuyLead has no category (MCAT=0) — cannot route to suppliers.")
            if "MCAT_MISMATCH" in _flag_ids:
                _mf = next((f for f in r["rca_flags"] if f["flag_id"] == "MCAT_MISMATCH"), None)
                _conf = f"{_mf['confidence']:.0%}" if _mf else ""
                rca_sentences.append(f"Category mismatch ({_conf} confidence) — wrong suppliers will receive lead.")
            if "NO_BUYER_FILLED_ISQS" in _flag_ids:
                _isq = bl.get("available_isq_at_approval", 0) or 0
                rca_sentences.append(f"Buyer filled 0/{_isq} ISQ questions — specs are system-predicted, low reliability.")
            if "MISSING_ISQ_FIELDS" in _flag_ids:
                _mf2 = next((f for f in r["rca_flags"] if f["flag_id"] == "MISSING_ISQ_FIELDS"), None)
                _m3  = _re.search(r"Completeness: (\d+/\d+)", _mf2["evidence"]) if _mf2 else None
                rca_sentences.append(f"ISQ incomplete ({_m3.group(1) if _m3 else 'partial'}) — suppliers lack product details.")
            if "TITLE_TOO_LONG" in _flag_ids:
                rca_sentences.append("Title too long — reduces search relevance.")
            if "MOSTLY_PREDICTED_ISQS" in _flag_ids:
                rca_sentences.append("Most spec values are model predictions — may be inaccurate.")
            if "BUYER_B2C_VS_PREDICTED_B2B" in _flag_ids:
                rca_sentences.append("Buyer intent is personal (B2C) but routed as B2B — wrong audience.")
            if "SUSPICIOUS_AUTO_ISQ_VALUES" in _flag_ids:
                rca_sentences.append("Auto-filled spec values contradict title.")
            if not rca_sentences and r["rca_flags"]:
                rca_sentences.append(r["rca_flags"][0]["evidence"][:120])
            if not rca_sentences:
                rca_sentences.append("All quality checks passed — ready for routing.")

            rca_explanation = " ".join(rca_sentences[:2])

            html_rows += f"""
            <tr style="border-bottom:1px solid #eee">
              <td style="padding:8px;font-size:12px;color:#555">{r['buylead_id']}</td>
              <td style="padding:8px;font-size:13px;max-width:180px;overflow:hidden;text-overflow:ellipsis">{r['title'][:50]}</td>
              <td style="padding:8px;font-size:12px">{r['mapped_mcat']['name'][:25]}</td>
              <td style="padding:8px;text-align:center">
                <span style="font-size:18px;font-weight:700;color:{color}">{r['quality_score']}</span>
              </td>
              <td style="padding:8px;text-align:center">
                <span style="background:{color};color:white;padding:2px 8px;border-radius:12px;font-size:11px">{band}</span>
              </td>
              <td style="padding:8px;font-size:11px;color:#666;max-width:200px">{flags_summary or "✅ Clean"}</td>
              <td style="padding:8px;font-size:11px;color:{rca_color};max-width:180px;line-height:1.6">{action_text}</td>
              <td style="padding:8px;font-size:11px;color:#444;max-width:250px;line-height:1.6">{rca_explanation}</td>
            </tr>"""
        except Exception as e:
            import traceback as _tb
            print(f"    ❌ Failed: {e}")
            print(_tb.format_exc()[:400])
            html_rows += f"""
            <tr style="border-bottom:1px solid #eee;background:#fff8f8">
              <td colspan="7" style="padding:8px;color:#e53e3e;font-size:12px">
                ❌ {bl_id} ({str(bl.get('eto_ofr_title',''))[:30]}): {str(e)[:150]}
              </td>
            </tr>"""

    # Summary bar
    total       = len(raw)
    avg_score   = round(sum(r.get("quality_score",0) for r in results) / max(len(results),1), 1)
    summary_html = f"""
    <div style="display:flex;gap:12px;flex-wrap:wrap;margin-bottom:16px">
      <div style="padding:12px 20px;background:#f5f5f5;border-radius:8px;text-align:center">
        <div style="font-size:24px;font-weight:700">{total}</div>
        <div style="font-size:12px;color:#666">Total BuyLeads</div>
      </div>
      <div style="padding:12px 20px;background:#f5f5f5;border-radius:8px;text-align:center">
        <div style="font-size:24px;font-weight:700;color:{'#2d8a4e' if avg_score>=60 else '#854f0b' if avg_score>=40 else '#a32d2d'}">{avg_score}</div>
        <div style="font-size:12px;color:#666">Avg Score</div>
      </div>
      {''.join(f'<div style="padding:12px 20px;background:#f5f5f5;border-radius:8px;text-align:center"><div style="font-size:24px;font-weight:700;color:{BAND_COLOR.get(b,chr(35)+chr(52)+chr(52)+chr(52))}">{cnt}</div><div style="font-size:12px;color:#666">{b}</div></div>' for b,cnt in band_counts.items() if cnt > 0)}
    </div>"""

    table_html = f"""
    {summary_html}
    <div style="overflow-x:auto">
    <table style="width:100%;border-collapse:collapse;font-family:sans-serif">
      <thead>
        <tr style="background:#f0f0f0;text-align:left">
          <th style="padding:10px 8px;font-size:12px">ID</th>
          <th style="padding:10px 8px;font-size:12px">Title</th>
          <th style="padding:10px 8px;font-size:12px">MCAT</th>
          <th style="padding:10px 8px;font-size:12px;text-align:center">Score</th>
          <th style="padding:10px 8px;font-size:12px;text-align:center">Band</th>
          <th style="padding:10px 8px;font-size:12px">Top Flags</th>
          <th style="padding:10px 8px;font-size:12px;min-width:180px">Suggested Actions</th>
          <th style="padding:10px 8px;font-size:12px;min-width:250px">RCA (Root Cause)</th>
        </tr>
      </thead>
      <tbody>{html_rows}</tbody>
    </table>
    </div>"""

    # Save results JSON for download
    output_path = "/content/drive/MyDrive/buylead_rca/outputs/batch_results.json"
    try:
        import os
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        download_path = output_path
    except Exception:
        import tempfile
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".json", mode="w")
        json.dump(results, tmp, indent=2, ensure_ascii=False)
        tmp.close()
        download_path = tmp.name

    return table_html, download_path


# ══════════════════════════════════════════════════════════════
# GRADIO UI
# ══════════════════════════════════════════════════════════════
EXAMPLES = [
    ["Plastic 4dx JCB Excavator Toy", "", "6032", "JCB Excavator",
     "Material:Plastic | Color:Yellow | Quantity:50 | Quantity Unit:Piece",
     "214 | 214 | 212 | 213", "5"],
    ["Cotton Carry Bags", "", "25024", "Cloth Bags",
     "Bag Type:Printed | Material:Cotton | Quantity:100 | Quantity Unit:Piece | Probable Requirement Type:Business Use",
     "214 | 214 | 212 | 213 | 1000", "5"],
    ["Tiger King Xxl African Size Capsule", "", "655167", "Penis Enlargement Product",
     "Form:Capsule | Packaging Type:Bottle | Quantity:1 | Quantity Unit:Pack",
     "230 | 230 | 1 | 2", "3"],
    ["bolero engine head", "", "182561", "Car Cylinder Head",
     "Preferred Brand:Bolero sle power +",
     "1", "2"],
]

with gr.Blocks(
    title="BuyLead Quality RCA Agent",
    theme=gr.themes.Soft(primary_hue="teal"),
    css="""
    .gradio-container { max-width: 1400px !important; }
    .tab-nav button { font-size: 14px !important; }
    """
) as demo:

    gr.HTML("""
    <div style="text-align:center;padding:24px 0 12px;border-bottom:1px solid #e0e0e0;margin-bottom:20px">
      <h1 style="font-size:26px;margin:0;font-weight:700">🔍 BuyLead Quality RCA Agent</h1>
      <p style="color:#666;margin:8px 0 0;font-size:14px">
        IndiaMART Hackathon &nbsp;·&nbsp; Rule Engine + MCAT Ranker + LLM Gateway
        &nbsp;·&nbsp; Explainable Scoring
      </p>
      <p style="color:#888;font-size:12px;margin:4px 0 0">
        Pipeline: Rule Engine → MCAT Ranker (98k MCATs) → Scorer → LLM Narrative
      </p>
    </div>
    """)

    with gr.Tabs():

        # ── Tab 1: Single BuyLead ─────────────────────────────
        with gr.TabItem("🔍 Single BuyLead"):
            with gr.Row():
                with gr.Column(scale=1, min_width=360):
                    gr.Markdown("### Input")
                    title_in  = gr.Textbox(label="Title *", placeholder="e.g. Plastic JCB Excavator Toy")
                    desc_in   = gr.Textbox(label="Description", lines=2,
                                           placeholder="Leave empty to test auto-generation")
                    with gr.Row():
                        mcat_id_in   = gr.Textbox(label="MCAT ID",   value="6032",        scale=1)
                        mcat_name_in = gr.Textbox(label="MCAT Name", value="JCB Excavator",scale=2)
                    specs_in  = gr.Textbox(
                        label="Specifications (Key:Value | Key:Value)",
                        lines=3,
                        value="Material:Plastic | Color:Yellow | Quantity:50 | Quantity Unit:Piece"
                    )
                    src_in    = gr.Textbox(
                        label="Attribute Source (pipe-separated codes)",
                        value="214 | 214 | 212 | 213",
                        info="1-205=Buyer, 207-221=Agent, 206/230-260=Auto, 1000=Predicted"
                    )
                    isq_in    = gr.Textbox(
                        label="ISQs shown at approval",
                        value="5",
                        info="How many ISQ questions were shown to buyer"
                    )
                    btn_single = gr.Button("🔍 Analyze", variant="primary", size="lg")
                    gr.Markdown("**Quick examples:**")
                    gr.Examples(
                        examples=EXAMPLES,
                        inputs=[title_in, desc_in, mcat_id_in, mcat_name_in,
                                specs_in, src_in, isq_in],
                        label="",
                    )

                with gr.Column(scale=1, min_width=400):
                    gr.Markdown("### Results")
                    status_out = gr.HTML(
                        value="<p style='color:#aaa;font-size:13px;padding:8px'>Enter BuyLead details and click Analyze.</p>"
                    )
                    with gr.Tabs():
                        with gr.TabItem("📊 Score"):
                            score_out = gr.HTML()
                        with gr.TabItem("⚠️ Flags"):
                            flags_out = gr.HTML()
                        with gr.TabItem("🏷️ MCAT"):
                            mcat_out  = gr.HTML()
                        with gr.TabItem("💬 Narrative"):
                            narr_out  = gr.HTML()

            LOADING_HTML = (
                "<div style='padding:14px;background:#f0f7ff;border-radius:8px;"
                "border-left:4px solid #3182ce;margin-bottom:8px'>"
                "<div style='display:flex;align-items:center;gap:10px'>"
                "<svg width='20' height='20' viewBox='0 0 50 50' style='animation:spin 1s linear infinite'>"
                "<circle cx='25' cy='25' r='20' fill='none' stroke='#3182ce' stroke-width='5' "
                "stroke-dasharray='80' stroke-dashoffset='60'/></svg>"
                "<span style='font-size:13px;color:#3182ce;font-weight:600'>Analyzing BuyLead...</span></div>"
                "<div style='font-size:12px;color:#555;margin-top:6px;margin-left:30px'>"
                "Stage 1: Rule Engine &nbsp;→&nbsp; Stage 2: MCAT Ranker &nbsp;→&nbsp; "
                "Stage 3: Scorer &nbsp;→&nbsp; Stage 4: LLM Narrative</div></div>"
                "<style>@keyframes spin{from{transform:rotate(0deg)}to{transform:rotate(360deg)}}"
                "svg{transform-origin:center}</style>"
            )

            btn_single.click(
                fn=lambda: (LOADING_HTML, "", "", "", ""),
                inputs=[],
                outputs=[status_out, score_out, flags_out, mcat_out, narr_out],
                queue=False,
            ).then(
                fn=run_single,
                inputs=[title_in, desc_in, mcat_id_in, mcat_name_in,
                        specs_in, src_in, isq_in],
                outputs=[score_out, flags_out, mcat_out, narr_out],
            ).then(
                fn=lambda: "<p style='color:#2d8a4e;font-size:12px;padding:4px'>✅ Analysis complete</p>",
                inputs=[],
                outputs=[status_out],
            )

        # ── Tab 2: Batch JSON Upload ──────────────────────────
        with gr.TabItem("📂 Batch JSON Upload"):
            gr.Markdown("""
            ### Upload BuyLeads JSON file
            Upload a `.json` file containing an array of BuyLead objects.
            Each BuyLead will be analyzed and results shown in a table.

            **Required fields:** `display_id`, `eto_ofr_title`, `eto_ofr_mcat_id`, `specifications`

            **Optional:** `glcat_mcat_name`, `eto_ofr_desc`, `attribute_source`, `available_isq_at_approval`
            """)

            with gr.Row():
                file_in  = gr.File(label="Upload buyLeads.json", file_types=[".json"])
                btn_batch = gr.Button("🚀 Run Batch Analysis", variant="primary", size="lg")

            batch_out    = gr.HTML(label="Results Table")
            download_out = gr.File(label="Download Results JSON")

            btn_batch.click(
                fn=run_batch,
                inputs=[file_in],
                outputs=[batch_out, download_out],
            )

            gr.Markdown("""
            **Output columns:**
            - **Score** — quality score 0-100
            - **Band** — EXCELLENT / GOOD / MEDIUM / POOR / CRITICAL
            - **Top Flags** — most severe issues found (flag_id + severity initial)

            Full results saved to `/content/drive/MyDrive/buylead_rca/outputs/batch_results.json`
            """)

    gr.HTML("""
    <div style="text-align:center;padding:16px;color:#aaa;font-size:11px;border-top:1px solid #eee;margin-top:20px">
      IndiaMART BuyLead RCA Agent &nbsp;·&nbsp;
      Rule Engine (9 rules) + MCAT Ranker (98k MCATs, embeddings + LLM) +
      Scorer (8 checks, 100 pts) + LLM Narrative (Gateway)
    </div>
    """)

# ── Launch ────────────────────────────────────────────────────
print("🚀 Launching Gradio demo...")
demo.launch(share=True)

/tmp/ipykernel_1157/2237000050.py:417: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1157/2237000050.py:417: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


🚀 Launching Gradio demo...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e0cc676deddebe3b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [35]:
# ── CELL: Test LLM Gateway Connection ────────────────────────
import openai, os, json

# Step 1: Check key is set
key = os.environ.get("LLM_API_KEY", "LLM_API_KEY")
if not key:
    print("❌ LLM_API_KEY not set")
    print("   Run: os.environ['LLM_API_KEY'] = 'sk-xxx'")
else:
    print(f"✅ LLM_API_KEY found: {key[:8]}...{key[-4:]}")

# Step 2: Create client
try:
    client = openai.OpenAI(
        api_key  = key,
        base_url = "https://imllm.intermesh.net/v1",
    )
    print("✅ OpenAI client created")
except Exception as e:
    print(f"❌ Client creation failed: {e}")

# Step 3: Simple ping — send one message
print("\n🔄 Sending test message to gateway...")
try:
    response = client.chat.completions.create(
        model    = "anthropic/claude-sonnet-4-6",
        messages = [
            {"role": "user", "content": "Reply with exactly this JSON: {\"status\": \"connected\"}"}
        ],
        max_tokens  = 20,
        temperature = 0,
    )
    reply = response.choices[0].message.content.strip()
    print(f"✅ Gateway responded: {reply}")

    # Step 4: Check JSON response
    try:
        parsed = json.loads(reply.replace("```json","").replace("```","").strip())
        print(f"✅ JSON parsed: {parsed}")
        print(f"\n🎉 LLM Gateway is WORKING")
        print(f"   Model : qwen/qwen3-32b")
        print(f"   URL   : https://imllm.intermesh.net/v1")
    except:
        print(f"⚠️  Connected but response not JSON: {reply}")

except openai.AuthenticationError:
    print("❌ Authentication failed — check your API key")
except openai.APIConnectionError as e:
    print(f"❌ Cannot reach gateway: {e}")
    print("   Check VPN / network access to imllm.intermesh.net")
except Exception as e:
    print(f"❌ Gateway error: {e}")

✅ LLM_API_KEY found: LLM_API_..._KEY
✅ OpenAI client created

🔄 Sending test message to gateway...
❌ Authentication failed — check your API key


In [36]:
import os

# Set your actual key here
os.environ["LLM_API_KEY"] = "sk-wfXhIKbRco6HyjrgOfklwg"   # ← paste your real key

# Verify
key = os.environ.get("LLM_API_KEY", "")
print(f"Key set: {key[:8]}...{key[-4:]}")
print(f"Length: {len(key)} chars")

Key set: sk-wfXhI...klwg
Length: 25 chars


In [37]:
import openai, os, json

key = os.environ.get("LLM_API_KEY", "")   # ← empty string fallback, not "LLM_API_KEY"

if not key:
    print("❌ Key not set — run: os.environ['LLM_API_KEY'] = 'your_key'")
else:
    print(f"✅ Key found: {key[:8]}...{key[-4:]} ({len(key)} chars)")

    client = openai.OpenAI(
        api_key  = key,
        base_url = "https://imllm.intermesh.net/v1",
    )

    response = client.chat.completions.create(
        model    = "anthropic/claude-sonnet-4-6",
        messages = [{"role": "user", "content": "say hello"}],
        max_tokens = 10,
    )
    print(f"✅ Gateway working: {response.choices[0].message.content.strip()}")

✅ Key found: sk-wfXhI...klwg (25 chars)
✅ Gateway working: Hello! 👋 How are you doing
